# Optimizer Benchmark Analysis

Loads CSV I/II/III and Wandb run histories to produce paper-ready plots.

In [ ]:
# ── Load CSVs (needed for sections 2+) ───────────────────────────────────

RESULTS_DIR = r"C:\Users\mtnbt\My Drive (mtnbt123@gmail.com)\University\Research\Projects\2025-JailbreakToolbox\tropt\scripts\opt-bench\results"

# import sys
# project_dir = '/home/sharifm/students/matanbentov/TROPT'
# sys.path.append(project_dir)
# project_dir = '/home/sharifm/students/matanbentov/TROPT'
# os.chdir(project_dir)

# Add the project directory to the sys.path to ensure Python imports from there
# sys.path.append(project_dir)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import wandb

# ── Paths & constants ──────────────────────────────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)

WANDB_ENTITY  = "matanbt"
WANDB_PROJECT = "tropt-optbench"

# Paper-style aesthetics
sns.set_theme(style="whitegrid", font_scale=1.2)
PALETTE = "tab20"
FIG_DPI = 150

# ── Human-readable optimizer labels (used by every plot/table below) ──
# Keys are the raw `optimizer_name` values stored in wandb configs / CSVs.
# Unknown names fall through to their raw form.
OPTIMIZER_LABELS: dict[str, str] = {
    "gcg":           "GCG",
    "nanogcg":       "nanoGCG",
    "gcgplus_rand":  "GCG+ (rand init)",
    "mac":           "MAC",
    "pal":           "PAL",
    "ral":           "RAL",
    "arca":          "ARCA",
    "autoprompt":    "AutoPrompt",
    "hotflip":       "HotFlip",
    "beast":         "BEAST",
    "pez":           "PEZ",
    "gbda":          "GBDA",
    "qcg":           "QCG",
    "random_search": "Rand. Search",
    "adv_decoding":  "AdvDecoding",
    "gaslite":       "GASLITE",
    "gaslite2":      "GASLITE-2",
    "soft_prompt":   "Soft Prompt",
}

def optimizer_label(name: str) -> str:
    return OPTIMIZER_LABELS.get(name, name)

# Default optimizer color map; the CSV loader cell overrides this with a
# stable palette once it knows the full optimizer set. Defining a fallback
# here lets the dynamics cells run without the CSV loader.
OPTIMIZER_COLORS: dict[str, tuple] = {}

def optimizer_color(name: str):
    return OPTIMIZER_COLORS.get(name, (0.5, 0.5, 0.5))

VARIANT_LABELS: dict[str, str] = {
    "gcg_vanilla":         "Base",
    "gcg_cw":              "+CW Loss",
    "gcg_flrt_clamp":      "+CE-Clamping Loss",
    "gcg_flrt_distill":    "+Distil Loss",
    "gcg_steering":        "+Steering Loss",
    "gcg_attn_hijack":     "+Attn-Hijack Loss",
    "gcg_begging_init":    "+Hot Init",
    "gcg_jailbroken_target": "+Jailbroken Target",
    "gcg_prs_template":    "+Jailbreak Template",
}

def variant_label(name: str) -> str:
    return VARIANT_LABELS.get(name, name)


# 1  Exp1: Optimization Dynamics

Line plot of loss over time per optimizer, for a chosen (model, message).
Shading = std across seeds. Dashed line = soft_prompt lower bound.

Only requires WandB access (no CSVs needed).

In [ ]:
# ── Configuration ─────────────────────────────────────────────
CHOSEN_MSG_ID = 8
# CHOSEN_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"  # set to your model
CHOSEN_MODEL = "google/gemma-3-12b-it"
# CHOSEN_MODEL = "Qwen/Qwen3-8B"

# Optimizers to drop from every plot in this subsection (dynamics + highlighted).
# Applied at load time, so neither `histories` nor `lb_histories` see them.
DYNAMICS_BLACKLIST_OPTIMIZERS: list[str] = [
    "soft_prompt"
]


api = wandb.Api()
# Filter server-side: api.runs() returns lazy runs and `run.config` comes back
# empty until wandb is forced to hydrate it. Matching on config.* forces that
# hydration, so the cfg.get(...) reads below return real values. Also covers
# both regular and soft_prompt runs in a single query.
runs = api.runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}",
    filters={
        "state": "finished",
        "config.run_type": "optbench_whitebox",
        "config.model_name": CHOSEN_MODEL,
        "config.msg_id": CHOSEN_MSG_ID,
    },
)

# Keys to fetch from wandb (always include _runtime for time axis).
# NOTE: Do NOT include "step" — it is not logged; wandb provides "_step" automatically.
HISTORY_KEYS = ["loss", "best_loss", "total_models_stats/total_flops", "_runtime"]

# Collect per-step histories grouped by (optimizer, seed)
histories: dict[str, dict[int, pd.DataFrame]] = {}
lb_histories: dict[int, pd.DataFrame] = {}   # soft_prompt lower bound

for run in runs:
    # Belt-and-braces: if config still came back empty for any reason, force load.
    cfg = run.config
    if not cfg:
        run.load(force=True)
        cfg = run.config

    opt  = cfg.get("optimizer_name", "unknown")
    if opt in DYNAMICS_BLACKLIST_OPTIMIZERS:
        continue
    seed = cfg.get("seed", 0)
    is_soft = cfg.get("is_soft", False)

    try:
        hist = run.history(keys=HISTORY_KEYS, x_axis="_step", pandas=True)
    except Exception:
        hist = run.history(pandas=True)
    if hist.empty:
        continue
    # Normalise column names: _step -> step
    if "_step" in hist.columns:
        hist = hist.rename(columns={"_step": "step"})
    if "total_models_stats/total_flops" in hist.columns:
        hist = hist.rename(columns={"total_models_stats/total_flops": "flops"})
    if "_runtime" in hist.columns:
        hist = hist.rename(columns={"_runtime": "time"})
    if is_soft:
        lb_histories[seed] = hist
    else:
        histories.setdefault(opt, {})[seed] = hist

print(f"Loaded {sum(len(v) for v in histories.values())} runs for {len(histories)} optimizers")
if DYNAMICS_BLACKLIST_OPTIMIZERS:
    print(f"Blacklist skipped: {DYNAMICS_BLACKLIST_OPTIMIZERS}")
if lb_histories:
    print(f"Loaded {len(lb_histories)} soft_prompt (lower-bound) runs")


In [ ]:
# ── Plot configuration ─────────────────────────────────────────────────────
X_AXIS = "flops"      # "time" (wall-clock seconds) or "flops"
Y_AXIS = "best_loss"  # "loss" (per-step) or "best_loss" (running best)

fig, ax = plt.subplots(figsize=(10, 5))

for opt_name, seed_hists in histories.items():
    color = optimizer_color(opt_name)
    # Resolve x column: prefer requested axis, fall back to step
    sample_cols = next(iter(seed_hists.values())).columns
    if X_AXIS in sample_cols:
        x_col = X_AXIS
    elif "step" in sample_cols:
        x_col = "step"
    else:
        x_col = sample_cols[0]
    # Align on common x grid by interpolating
    all_x = sorted(set(x for h in seed_hists.values() for x in h[x_col].dropna()))
    interp_losses = []
    for hist in seed_hists.values():
        valid = hist[[x_col, Y_AXIS]].dropna()
        if len(valid) < 2:
            continue
        interp_losses.append(np.interp(all_x, valid[x_col], valid[Y_AXIS]))
    if not interp_losses:
        continue
    arr = np.array(interp_losses)
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    ax.plot(all_x, mean, label=optimizer_label(opt_name), color=color)
    ax.fill_between(all_x, mean - std, mean + std, alpha=0.15, color=color)

# Lower-bound dashed line
if lb_histories:
    sample_cols = next(iter(lb_histories.values())).columns
    x_col = X_AXIS if X_AXIS in sample_cols else "step"
    lb_losses = []
    all_x_lb = sorted(set(x for h in lb_histories.values() for x in h[x_col].dropna()))
    for hist in lb_histories.values():
        valid = hist[[x_col, Y_AXIS]].dropna()
        if len(valid) < 2:
            continue
        lb_losses.append(np.interp(all_x_lb, valid[x_col], valid[Y_AXIS]))
    if lb_losses:
        lb_mean = np.array(lb_losses).mean(axis=0)
        ax.plot(all_x_lb, lb_mean, linestyle="--", color="black", linewidth=1.5, label="soft_prompt (lower bound)")

X_LABELS = {"flops": "FLOPs", "time": "Wall-clock Time (s)"}
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(X_LABELS.get(X_AXIS, X_AXIS))
ax.set_ylabel(Y_AXIS.replace("_", " ").title())
ax.set_title(f"Optimization Dynamics — model: {CHOSEN_MODEL.split('/')[-1]}, msg: {CHOSEN_MSG_ID}")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(f"{RESULTS_DIR}/fig_dynamics_m{CHOSEN_MSG_ID}.pdf", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

### 1.1  Dynamics — Highlighted Methods

Same as the plot above, but only five methods (GCG, HotFlip, MAC, PAL, RAL)
are drawn in color; the rest are thin grey lines so the headline methods stand
out. FLOPs axis clipped to ≥ 10^14 to cut the noisy early region.


In [ ]:
# ── Highlighted dynamics plot ───────────────────────────────────
# Only these five get color + std band + legend entry. Everything else is
# drawn thin-grey behind them. Uses the same `histories` populated above,
# so DYNAMICS_BLACKLIST_OPTIMIZERS applies here too.
HIGHLIGHTED = {"gcg", "hotflip", "mac", "pal",  "random_search"}  # "ral",
# X_MIN_FLOPS = 0
X_MIN_FLOPS = 2e15  # for log axis
SHOW_STD_BAND = False  # toggle std shadow on/off

X_AXIS = "flops"
Y_AXIS = "best_loss"

# Curated palette (colorblind-friendly, print-safe). Override per-optimizer so
# the plot looks consistent regardless of what optimizer_color() returns.
HIGHLIGHT_COLORS = {
    "gcg":           "#1F77B4",  # deep blue
    "hotflip":       "#D1495B",  # muted crimson
    "mac":           "#2A9D8F",  # teal
    "pal":           "#E9C46A",  # warm gold
    "ral":           "#8E44AD",  # purple
    "random_search": "#8E44AD",  # purple (same as ral, in case of naming mismatch)
}

# Scoped style: Times Roman only for this plot, reverts after the `with` block.
with plt.rc_context({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.9,
    "ytick.major.width": 0.9,
    "xtick.minor.width": 0.7,
    "ytick.minor.width": 0.7,
    "axes.spines.top": False,
    "axes.spines.right": False,
}):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))

    # First pass: compute the global max FLOPs across highlighted optimizers,
    # so we can extrapolate short runs by holding their last value.
    global_max_x = 0.0
    for opt_name, seed_hists in histories.items():
        if opt_name not in HIGHLIGHTED:
            continue
        sample_cols = next(iter(seed_hists.values())).columns
        x_col = (X_AXIS if X_AXIS in sample_cols
                 else "step" if "step" in sample_cols else sample_cols[0])
        for hist in seed_hists.values():
            valid = hist[[x_col, Y_AXIS]].dropna()
            valid = valid[valid[x_col] >= X_MIN_FLOPS]
            if len(valid) >= 2:
                global_max_x = max(global_max_x, float(valid[x_col].max()))

    # Collected per-optimizer plot data so we can order the legend by final loss.
    highlighted_plots = {}  # opt_name -> dict(x, mean, std, color, final_loss)

    # Two passes so highlighted lines draw on top of the grey background lines.
    for highlighted_pass in (False, True):
        for opt_name, seed_hists in histories.items():
            if (opt_name in HIGHLIGHTED) != highlighted_pass:
                continue
            sample_cols = next(iter(seed_hists.values())).columns
            if X_AXIS in sample_cols:
                x_col = X_AXIS
            elif "step" in sample_cols:
                x_col = "step"
            else:
                x_col = sample_cols[0]
            all_x = sorted({x for h in seed_hists.values()
                            for x in h[x_col].dropna() if x >= X_MIN_FLOPS})
            if len(all_x) < 2:
                continue

            # For highlighted lines, extend the x grid to global_max_x so short
            # runs get extrapolated (np.interp holds the endpoint value by default).
            if highlighted_pass and global_max_x > all_x[-1]:
                all_x = all_x + [global_max_x]

            interp_losses = []
            for hist in seed_hists.values():
                valid = hist[[x_col, Y_AXIS]].dropna()
                valid = valid[valid[x_col] >= X_MIN_FLOPS]
                if len(valid) < 2:
                    continue
                # np.interp holds valid[Y_AXIS].iloc[-1] for x > valid[x_col].max(),
                # which is exactly the "hold last value" extrapolation we want.
                interp_losses.append(np.interp(all_x, valid[x_col], valid[Y_AXIS]))
            if not interp_losses:
                continue
            arr = np.array(interp_losses)
            mean = arr.mean(axis=0)
            if highlighted_pass:
                # Fall back to a palette default (purple) rather than optimizer_color(),
                # which may return grey and cause the line to blend with the background.
                color = HIGHLIGHT_COLORS.get(opt_name, "#8E44AD")
                std = arr.std(axis=0)
                highlighted_plots[opt_name] = {
                    "x": all_x, "mean": mean, "std": std,
                    "color": color, "final_loss": float(mean[-1]),
                }
            else:
                ax.plot(all_x, mean, color="0.55", linewidth=1.1,
                        alpha=0.65, zorder=1)

    # Draw highlighted lines in legend order (best final loss first).
    ordered = sorted(highlighted_plots.items(), key=lambda kv: kv[1]["final_loss"])
    for opt_name, d in ordered:
        ax.plot(d["x"], d["mean"], label=optimizer_label(opt_name), color=d["color"],
                linewidth=2.2, solid_capstyle="round", zorder=3)
        if SHOW_STD_BAND:
            ax.fill_between(d["x"], d["mean"] - d["std"], d["mean"] + d["std"],
                            alpha=0.15, color=d["color"], zorder=2)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(left=X_MIN_FLOPS)
    ax.set_xlabel("FLOPs")
    ax.set_ylabel("Loss (CE)")
    # ax.set_ylabel(Y_AXIS.replace("_", " ").title())

    ax.grid(True, which="major", linestyle="-", linewidth=0.4,
            color="0.88", zorder=0)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.3,
            color="0.92", zorder=0)
    ax.tick_params(which="both", direction="out")

    leg = ax.legend(loc="lower left", frameon=True, framealpha=0.92,
                    edgecolor="0.8", fancybox=False, borderpad=0.55,
                    handlelength=1.8, handletextpad=0.6, labelspacing=0.35)
    leg.get_frame().set_linewidth(0.6)

    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_dynamics_highlighted_m{CHOSEN_MSG_ID}.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Horizontal bar chart: final best_loss per optimizer ────────────────────
# Uses the same `histories` dict from the download cell above.
# Each bar = mean of final best_loss across seeds; error bar = std across seeds.

final_losses = {}
for opt_name, seed_hists in histories.items():
    finals = [h["best_loss"].iloc[-1] for h in seed_hists.values() if not h.empty]
    if finals:
        final_losses[opt_name] = finals

opts = sorted(final_losses.keys(), key=lambda o: np.mean(final_losses[o]))
means = [np.mean(final_losses[o]) for o in opts]
stds  = [np.std(final_losses[o]) for o in opts]

fig, ax = plt.subplots(figsize=(8, max(4, len(opts) * 0.45)))
colors = [optimizer_color(o) for o in opts]
ax.barh(opts, means, xerr=stds, capsize=4, color=colors, edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("Best Loss (log scale, lower is better)")
ax.set_title(f"Final Best Loss — model: {CHOSEN_MODEL.split('/')[-1]}, msg: {CHOSEN_MSG_ID}")
ax.invert_yaxis()

# Add lower-bound line if available
if lb_histories:
    lb_finals = [h["best_loss"].iloc[-1] for h in lb_histories.values() if not h.empty]
    if lb_finals:
        lb_mean = np.mean(lb_finals)
        ax.axvline(lb_mean, color="black", linestyle="--", linewidth=1.5, label="soft_prompt (lower bound)")
        ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(f"{RESULTS_DIR}/fig_bar_best_loss_m{CHOSEN_MSG_ID}.pdf", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

### 1.2  NanoGCG vs TROPT GCG -- Dynamics

Loss-vs-time curves for `gcg` and `nanogcg` on the chosen (model, msg).
Reuses the `histories` dict from section 1, which is populated directly
from wandb and is *not* affected by `BLACKLIST_OPTIMIZERS` (only by
`DYNAMICS_BLACKLIST_OPTIMIZERS`, which excludes `soft_prompt`).

In [ ]:
# NanoGCG vs TROPT GCG: best_loss vs wall-clock time, mean +/- std across seeds.
# Reuses the `histories` dict from section 1; permissive wandb fallback when
# an optimizer is missing (different metric names / state / etc.).
PAIR        = ["nanogcg", "gcg"]            # legend / color order
PAIR_COLORS = {"gcg": "#1F4E79", "nanogcg": "#C0392B"}
PAIR_LABELS = {"gcg": "TROPT's GCG", "nanogcg": "nanoGCG's GCG"}
X_AXIS = "time"
Y_AXIS = "best_loss"


def _fetch_histories_for(opt_name: str) -> dict[int, pd.DataFrame]:
    """Permissive wandb fetch (no state filter, no key whitelist)."""
    out: dict[int, pd.DataFrame] = {}
    runs = list(api.runs(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}",
        filters={
            "config.optimizer_name": opt_name,
            "config.model_name":     CHOSEN_MODEL,
            "config.msg_id":         1,
        },
    ))
    print(f"  [{opt_name}] wandb returned {len(runs)} run(s) for "
          f"(model={CHOSEN_MODEL}, msg={CHOSEN_MSG_ID})")
    for run in runs:
        cfg = dict(run.config) if run.config else {}
        if not cfg:
            run.load(force=True)
            cfg = dict(run.config)
        seed = cfg.get("seed", len(out))
        hist = run.history(pandas=True)
        if hist.empty:
            print(f"    seed={seed} state={run.state}: empty history")
            continue
        # nanogcg logs both `_step` (auto) and `step` (explicit). Drop the
        # explicit one so renaming `_step -> step` does not produce duplicates.
        if "step" in hist.columns and "_step" in hist.columns:
            hist = hist.loc[:, ~((hist.columns == "step") &
                                 (hist.columns.duplicated(keep=False) == False))]
            # Above kept the first non-dup; simpler: drop the explicit `step`.
            hist = hist.drop(columns=["step"])
        rename = {"_step": "step", "_runtime": "time",
                  "total_models_stats/total_flops": "flops"}
        hist = hist.rename(columns={k: v for k, v in rename.items() if k in hist.columns})
        print(f"    seed={seed} state={run.state} rows={len(hist)} "
              f"cols={[c for c in hist.columns if not c.startswith('_')][:8]}")
        out[seed] = hist
    return out


pair_hists: dict[str, dict[int, pd.DataFrame]] = {}
for opt_name in PAIR:
    if opt_name in histories and histories[opt_name]:
        pair_hists[opt_name] = histories[opt_name]
        print(f"[{opt_name}] using {len(histories[opt_name])} run(s) from `histories`")
    else:
        print(f"[fallback] {opt_name} missing from `histories` -- fetching from wandb")
        fetched = _fetch_histories_for(opt_name)
        if fetched:
            pair_hists[opt_name] = fetched
        else:
            print(f"  [warn] no usable runs for {opt_name}")

with plt.rc_context({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize":   16,
    "axes.labelsize":   16,
    "xtick.labelsize":  12,
    "ytick.labelsize":  12,
    "legend.fontsize":  13,
    "axes.spines.top":   False,
    "axes.spines.right": False,
}):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for opt_name in PAIR:
        seed_hists = pair_hists.get(opt_name, {})
        if not seed_hists:
            continue
        sample_cols = next(iter(seed_hists.values())).columns
        x_col = X_AXIS if X_AXIS in sample_cols else "step"
        y_candidates = [Y_AXIS, "best_loss", "loss"]
        y_col = next((c for c in y_candidates if c in sample_cols), None)
        if y_col is None:
            print(f"  [warn] {opt_name}: no recognised loss column; got {list(sample_cols)}")
            continue
        all_x = sorted({x for h in seed_hists.values() for x in h[x_col].dropna()})
        curves = []
        for hist in seed_hists.values():
            valid = hist[[x_col, y_col]].dropna()
            if len(valid) < 2:
                continue
            # Sort by x and collapse duplicate x values (keep min y) so
            # np.interp gets a monotone xp and cummin sees ordered data.
            valid = (valid.sort_values(x_col)
                          .groupby(x_col, as_index=False)[y_col].min())
            ys = valid[y_col].cummin() if y_col == "loss" else valid[y_col]
            curves.append(np.interp(all_x, valid[x_col], ys))
        if not curves:
            continue
        arr = np.array(curves)
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        c = PAIR_COLORS[opt_name]
        ax.plot(all_x, mean,
                label=f"{PAIR_LABELS[opt_name]} (n={len(curves)})",
                color=c, linewidth=2.2)
        ax.fill_between(all_x, mean - std, mean + std, alpha=0.18, color=c)

    ax.set_yscale("log")
    ax.set_xlabel("Wall-clock Time (s)")
    ax.set_ylabel("Best Loss (CE)")
    ax.grid(True, which="both", linestyle=":", linewidth=0.4, color="0.85", zorder=0)
    ax.legend(loc="upper right", frameon=True, framealpha=0.92, edgecolor="0.8")
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_dynamics_nanogcg_vs_gcg_m{CHOSEN_MSG_ID}.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

# Load (Exp1 CSVs)

Loads CSV I/II/III from each per-model result file and concatenates them so
ranking / plots aggregate across all runs by default.

Comment out a line in `EXP1_MODELS` to exclude that model and inspect the
remaining ones in isolation.

In [ ]:
# ── Models to aggregate across ───────────────────────────────────────
# Each entry is the filename slug exp1.py used when writing its CSVs
# (i.e. `{RESULTS_DIR}/exp1_wb_{slug}_csv_{i,ii,iii}.csv`).
# Comment out any line to drop that model from the aggregated analysis.
# Missing CSVs are skipped with a warning rather than raising.
EXP1_MODELS = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "Qwen3-8B",
    "gemma-4-26B-A4B-it",
]

# ── Optimizers to drop from every analysis below ─────────────────────
# Applied once after concat, so bar/box/Friedman/LaTeX all see the same set.
# Useful for excluding partial / still-running optimizers that would otherwise
# shrink the Friedman full-coverage task count. Soft optimizers are excluded
# separately, per-cell, via the `is_soft` column (not this list).
BLACKLIST_OPTIMIZERS: list[str] = [
    "nanogcg",
    "soft_prompt",
    "gaslite2",
    "gcgplus_rand",
]

def _load_if_exists(path: str) -> pd.DataFrame | None:
    if not os.path.exists(path):
        print(f"  [skip] missing {path}")
        return None
    return pd.read_csv(path)

frames_i, frames_ii, frames_iii = [], [], []
loaded_models = []
for slug in EXP1_MODELS:
    prefix = f"{RESULTS_DIR}/exp1_wb_{slug}"
    ci   = _load_if_exists(f"{prefix}_csv_i.csv")
    cii  = _load_if_exists(f"{prefix}_csv_ii.csv")
    ciii = _load_if_exists(f"{prefix}_csv_iii.csv")
    if ci is None and cii is None and ciii is None:
        continue
    loaded_models.append(slug)
    if ci   is not None: frames_i.append(ci)
    if cii  is not None: frames_ii.append(cii)
    if ciii is not None: frames_iii.append(ciii)

def _concat(frames):
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

csv_i   = _concat(frames_i)
csv_ii  = _concat(frames_ii)
csv_iii = _concat(frames_iii)

if BLACKLIST_OPTIMIZERS:
    before = {"i": len(csv_i), "ii": len(csv_ii), "iii": len(csv_iii)}
    for name, df in (("csv_i", csv_i), ("csv_ii", csv_ii), ("csv_iii", csv_iii)):
        if not df.empty and "optimizer_name" in df.columns:
            df.drop(df.index[df["optimizer_name"].isin(BLACKLIST_OPTIMIZERS)],
                    inplace=True)
    print(f"Blacklist {BLACKLIST_OPTIMIZERS} dropped: "
          f"csv_i {before['i']-len(csv_i)}, csv_ii {before['ii']-len(csv_ii)}, "
          f"csv_iii {before['iii']-len(csv_iii)} rows")

# ── Stable optimizer → color map (shared by every plot below) ────────
# Keyed alphabetically so rerunning with a different subset keeps each
# optimizer on the same color. Falls back to grey for unknown names.
_all_opts = sorted({
    o for df in (csv_i, csv_ii, csv_iii)
    if not df.empty and "optimizer_name" in df.columns
    for o in df["optimizer_name"].unique()
})
_palette = sns.color_palette(PALETTE, max(len(_all_opts), 20))
OPTIMIZER_COLORS: dict[str, tuple] = {o: _palette[i] for i, o in enumerate(_all_opts)}

def optimizer_color(name: str):
    return OPTIMIZER_COLORS.get(name, (0.5, 0.5, 0.5))

print(f"\nLoaded models ({len(loaded_models)}/{len(EXP1_MODELS)}): {loaded_models}")
print(f"CSV I   rows: {len(csv_i):>6}"
      + (f"  ({csv_i['model_name'].nunique()} models, "
         f"{csv_i['optimizer_name'].nunique()} optimizers)" if not csv_i.empty else ""))
print(f"CSV II  rows: {len(csv_ii):>6}")
print(f"CSV III rows: {len(csv_iii):>6}")
print(f"Optimizer palette: {len(OPTIMIZER_COLORS)} stable colors assigned")
if not csv_i.empty:
    print("Rows per model (CSV I):")
    print(csv_i.groupby("model_name").size().to_string())

## 2  Average Loss per Optimizer (CSV I)

Average final loss across all seeds and instructions.

In [ ]:
def bar_plot(df, value_col, title, ylabel, output_file=None,
             exclude_soft=True):
    """Bar plot: avg value per optimizer, with standard-error error bars.

    Aggregates across every row after filtering — so each row (one seed on one
    message on one model) is treated as an IID sample. SE therefore *over*states
    precision if you care about task-level uncertainty; prefer the box plot
    (section 5) for that.
    """
    plot_df = df.copy()
    if exclude_soft and "is_soft" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_soft"].astype(bool)]

    agg = (plot_df.groupby("optimizer_name")[value_col]
           .agg(["mean", "std", "count"])
           .reset_index()
           .sort_values("mean"))
    agg["se"] = agg["std"] / np.sqrt(agg["count"])

    fig, ax = plt.subplots(figsize=(max(8, len(agg) * 0.7), 5))
    colors = [optimizer_color(o) for o in agg["optimizer_name"]]
    ax.bar([optimizer_label(o) for o in agg["optimizer_name"]], agg["mean"], yerr=agg["se"],
           capsize=4, color=colors, edgecolor="white")
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    ax.set_yscale("log")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return agg

agg_loss = bar_plot(csv_i, "best_loss", "Average Final Loss per Optimizer", "Loss (↓ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_loss.pdf")
agg_loss

## 2.5  Optimizer Ranking (Friedman + Performance Profiles)

Tasks = `(model, msg_id)` with seeds aggregated by mean. Reports:

- **Mean rank** per optimizer (Friedman-style; lower = better) + Friedman χ² test.
- **Critical Difference (CD)** at α=0.05 via Nemenyi post-hoc (Demšar 2006).
  Optimizers whose mean ranks differ by less than CD are **not** statistically
  distinguishable.
- **Performance profile** (Dolan–Moré): fraction of tasks where an optimizer is
  within a factor τ of the best optimizer on that task.

Soft optimizers are excluded from the ranking — they serve as a lower-bound
reference, not a competing method.


In [ ]:
from scipy import stats

METRIC = "best_loss"
LOWER_IS_BETTER = True
# False: task = (model, msg_id), scores averaged across seeds (paper default).
# True:  task = (model, msg_id, seed), each seed is its own Friedman block.
# Enabling this inflates N (→ tighter CD, stronger test) but lets a single
# lucky/unlucky seed tilt the ranking, and makes full-coverage stricter
# because missing seeds drop whole tasks.
TASK_INCLUDES_SEED = True

# Soft optimizers are excluded here (lower-bound reference, not a competitor).
# Global optimizer blacklist is already applied at load time.
df_rank = csv_i.copy()
if "is_soft" in df_rank.columns:
    df_rank = df_rank[~df_rank["is_soft"].astype(bool)]

_task_keys = ["model_name", "msg_id"] + (["seed"] if TASK_INCLUDES_SEED else [])
# One score per (optimizer, *task_keys): .mean() averages across seeds when
# TASK_INCLUDES_SEED is False, and is a no-op when it is True.
task_scores = (df_rank.groupby(["optimizer_name"] + _task_keys)[METRIC]
               .mean().reset_index())

# Raw pivot: rows = tasks, cols = optimizers. Any NaN means an optimizer did
# not run on that task; dropna keeps only tasks with full coverage.
score_mat_raw = task_scores.pivot_table(
    index=_task_keys, columns="optimizer_name", values=METRIC,
)
score_mat = score_mat_raw.dropna(axis=0)
print(f"Tasks observed:       {len(score_mat_raw)}")
print(f"Tasks w/ full cover:  {len(score_mat)}  (dropped {len(score_mat_raw) - len(score_mat)})")
print(f"Optimizers compared:  {score_mat.shape[1]}")

if len(score_mat_raw) > 0:
    cov = (score_mat_raw.count() / len(score_mat_raw)).sort_values()
    partial = cov[cov < 1.0]
    if len(partial):
        print("\nOptimizers missing from some tasks "
              "(add to BLACKLIST_OPTIMIZERS in the load cell to recover tasks):")
        for opt, c in partial.items():
            print(f"  {opt:<20s} {c*100:5.1f}% ({int(round(c*len(score_mat_raw)))}/{len(score_mat_raw)})")

if len(score_mat) < 3 or score_mat.shape[1] < 3:
    print(f"\nSkipping Friedman / CD: need N≥3 tasks and k≥3 optimizers "
          f"(have N={len(score_mat)}, k={score_mat.shape[1]}).")
else:
    # Per-task ranks (1 = best); ties get average rank
    ranks = score_mat.rank(axis=1, ascending=LOWER_IS_BETTER, method="average")
    mean_ranks = ranks.mean(axis=0).sort_values()
    print("\nMean ranks (lower = better):")
    print(mean_ranks.to_string(float_format="%.2f"))

    # Friedman omnibus test
    stat, p = stats.friedmanchisquare(*[score_mat[c].values for c in score_mat.columns])
    print(f"\nFriedman chi^2 = {stat:.2f}, p = {p:.2e}")

    # Nemenyi critical difference at alpha=0.05 (Demsar 2006, Table 5)
    _Q_ALPHA_05 = {
        2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031,
        9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268, 13: 3.313, 14: 3.354,
        15: 3.391, 16: 3.426, 17: 3.458, 18: 3.489, 19: 3.517, 20: 3.544,
    }
    k = score_mat.shape[1]
    N = len(score_mat)
    q_alpha = _Q_ALPHA_05.get(k, 3.544)  # 20-optimizer value as a safe upper bound
    CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))
    print(f"\nCritical Difference (alpha=0.05, k={k}, N={N}): CD = {CD:.3f}")
    if N < 5:
        print(f"  [warn] N={N} is small — CD is loose and test is underpowered.")

    # Mean-rank bar with shaded "within-CD-of-best" region
    fig, ax = plt.subplots(figsize=(9, max(3.5, k * 0.32)))
    y_pos = np.arange(len(mean_ranks))
    ax.barh(y_pos, mean_ranks.values,
            color=[optimizer_color(o) for o in mean_ranks.index], edgecolor="white",
            tick_label=[optimizer_label(o) for o in mean_ranks.index])
    ax.set_yticks(y_pos)
    ax.set_yticklabels(mean_ranks.index)
    ax.invert_yaxis()
    best = mean_ranks.iloc[0]
    ax.axvspan(best, best + CD, color="grey", alpha=0.18,
               label=f"within CD of best (CD={CD:.2f})")
    ax.set_xlabel("Mean rank (1 = best)")
    ax.set_title(f"Friedman ranking - {METRIC} (Friedman p={p:.1e}, N={N})")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_friedman_ranks.pdf", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [ ]:
# Dolan-More performance profile.
# r_{p,s} = score_p_s / best_p across optimizers s on problem p (>= 1).
# rho_s(tau) = fraction of problems where r_{p,s} <= tau.
# Requires the score_mat / mean_ranks built above.
# Task unit follows TASK_INCLUDES_SEED from the Friedman cell
# (seed-averaged by default, per-seed when True).
TAU_MAX_CAP = 1e3  # losses span many orders of magnitude; cap tau for readability.

if "score_mat" not in globals() or len(score_mat) < 2 or score_mat.shape[1] < 2:
    print("Performance profile needs at least 2 tasks and 2 optimizers — skipped.")
else:
    mat = score_mat.values  # tasks x optimizers
    if LOWER_IS_BETTER:
        best_per_task = mat.min(axis=1, keepdims=True)
        ratios = mat / np.maximum(best_per_task, 1e-12)
    else:
        best_per_task = mat.max(axis=1, keepdims=True)
        ratios = best_per_task / np.maximum(mat, 1e-12)

    tau_max = min(max(float(np.nanmax(ratios)), 2.0), TAU_MAX_CAP)
    taus = np.logspace(0, np.log10(tau_max), 300)

    fig, ax = plt.subplots(figsize=(9, 5))
    # Iterate in mean-rank order so legend matches the ranking plot
    order = mean_ranks.index if "mean_ranks" in globals() else score_mat.columns
    for opt in order:
        j = list(score_mat.columns).index(opt)
        rho = (ratios[:, j][:, None] <= taus[None, :]).mean(axis=0)
        ax.plot(taus, rho, label=optimizer_label(opt), color=optimizer_color(opt), linewidth=1.6)

    ax.set_xscale("log")
    ax.set_xlabel(r"$\tau$ (factor of best on each task)")
    ax.set_ylabel(r"$\rho_s(\tau)$ - fraction of tasks within $\tau\times$ best")
    _task_unit = "(model, msg, seed)" if TASK_INCLUDES_SEED else "(model, msg)"
    ax.set_title(f"Performance profile (Dolan-More) - {METRIC}  "
                 f"(N={len(score_mat)} {_task_unit} tasks, k={score_mat.shape[1]})")
    ax.set_ylim(0, 1.02)
    ax.set_xlim(1, tau_max)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_performance_profile.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [ ]:
# Per-model rank heatmap. Same per-task ranks as the Friedman cell, but
# averaged *within* each model instead of globally — so specialists that win
# on one model and lose on others show up as uneven rows (watch for e.g.
# one green cell + two red cells on the same optimizer).
# Uses score_mat_raw (pre-dropna) + na_option="keep" so models whose tasks
# got filtered out by the Friedman full-coverage check still appear.
# Task unit follows TASK_INCLUDES_SEED from the Friedman cell.
if "score_mat_raw" not in globals() or len(score_mat_raw) < 1 or score_mat_raw.shape[1] < 2:
    print("Rank heatmap needs score_mat_raw with tasks and >=2 optimizers - skipped.")
else:
    task_ranks = score_mat_raw.rank(
        axis=1, ascending=LOWER_IS_BETTER, method="average", na_option="keep"
    )
    tasks_per_model = score_mat_raw.groupby(level="model_name").size()

    # Rows = optimizers, cols = models, cell = mean rank on that model's tasks.
    heat = task_ranks.groupby(level="model_name").mean().T
    heat["_avg"] = heat.mean(axis=1)
    heat = heat.sort_values("_avg", ascending=True).drop(columns="_avg")
    pretty_cols = [c.split("/")[-1] for c in heat.columns]
    heat.index = [optimizer_label(o) for o in heat.index]

    fig, ax = plt.subplots(figsize=(max(6, heat.shape[1] * 2),
                                     max(4, heat.shape[0] * 0.5)))
    sns.heatmap(heat.set_axis(pretty_cols, axis=1),
                annot=True, fmt=".1f", cmap="RdYlGn_r",
                cbar_kws={"label": "Mean rank (1 = best on that model)"},
                ax=ax, linewidths=0.5, linecolor="white")
    task_str = ", ".join(f"{m.split('/')[-1]}={n}" for m, n in tasks_per_model.items())
    ax.set_xlabel("Model")
    ax.set_ylabel("Optimizer")
    _task_unit = "(msg, seed)" if TASK_INCLUDES_SEED else "msg"
    ax.set_title(f"Per-model mean rank - {METRIC}")
    print(f"({_task_unit} tasks per model: {task_str})")
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_rank_heatmap.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# Paper main-figure bar plot: avg rank for a curated subset of optimizers.
# A three-dot ellipsis in the middle of the x-axis signals that additional
# optimizers from the full ranking are omitted for visual clarity.
PAPER_FIG_WHITELIST = [
    "gcg", "pal", "qcg", "mac", 
    # "arca", "autoprompt",
    "hotflip", "beast", "pez", "gbda", "random_search",
]
N_LEFT_OF_GAP = 3  # bars left of the "..." ellipsis
SHOW_ERROR_BARS = False  # subtle SEM error bars on each rank
SHOW_RANK_VALUES = False  # numeric avg-rank annotation inside each bar

if "mean_ranks" not in globals():
    print("Need mean_ranks from the Friedman cell - skipped.")
else:
    shown = [o for o in PAPER_FIG_WHITELIST if o in mean_ranks.index]
    missing = [o for o in PAPER_FIG_WHITELIST if o not in mean_ranks.index]
    if missing:
        print(f"[warn] whitelist optimizers absent from mean_ranks: {missing}")

    shown.sort(key=lambda o: mean_ranks[o])  # best first
    ranks_shown = np.array([mean_ranks[o] for o in shown])

    # Per-optimizer SEM across tasks (uses `ranks` from the Friedman cell).
    if SHOW_ERROR_BARS and "ranks" in globals():
        rank_sem = np.array([
            ranks[o].std(ddof=1) / np.sqrt(len(ranks)) for o in shown
        ])
    else:
        rank_sem = None

    n_left = min(N_LEFT_OF_GAP, len(shown))
    GAP_W = 1.4
    xs = np.concatenate([
        np.arange(n_left, dtype=float),
        np.arange(n_left, len(shown), dtype=float) + GAP_W,
    ])

    # Blue strength encodes rank: best rank -> deepest blue, worst -> lightest.
    # Map ranks linearly to [0.35, 0.95] of the Blues colormap.
    blues = plt.get_cmap("Blues")
    rmin, rmax = ranks_shown.min(), ranks_shown.max()
    if rmax > rmin:
        norm = (ranks_shown - rmin) / (rmax - rmin)
    else:
        norm = np.zeros_like(ranks_shown)
    bar_colors = [blues(0.95 - 0.6 * t) for t in norm]

    # Times Roman to match paper font (falls back gracefully if absent).
    rc = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "axes.grid": False,
    }
    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(6, 2))
        ax.bar(xs, ranks_shown, width=0.78,
               color=bar_colors, edgecolor="white", linewidth=0.6, zorder=2)
        if rank_sem is not None:
            ax.errorbar(xs, ranks_shown, yerr=rank_sem,
                        fmt="none", ecolor="#333333", elinewidth=0.7,
                        capsize=2.0, capthick=0.7, alpha=0.55, zorder=4)

        # Numeric avg-rank inside each bar near the top (white for contrast).
        if SHOW_RANK_VALUES:
            inner_pad = ranks_shown.max() * 0.06
            for x, h in zip(xs, ranks_shown):
                ax.text(x, h - inner_pad, f"{h:.1f}",
                        ha="center", va="top", rotation=0,
                        fontsize=7.5, color="white", zorder=5)

        # Vertical name labels above each bar (above the error bar when shown)
        pad = ranks_shown.max() * 0.04
        label_tops = ranks_shown + (rank_sem if rank_sem is not None else 0.0)
        for x, opt, top in zip(xs, shown, label_tops):
            ax.text(x, top + pad, optimizer_label(opt),
                    ha="center", va="bottom", rotation=90,
                    fontsize=11, color="#222222",
                    fontfamily="Bahnschrift")

        # Three-dot ellipsis centered in the gap
        if 0 < n_left < len(shown):
            gx = (xs[n_left - 1] + xs[n_left]) / 2
            gy = ranks_shown.mean() * 0.55
            for dx in (-0.18, 0.0, 0.18):
                ax.plot(gx + dx, gy, marker="o", markersize=3.5,
                        color="#888888", zorder=3, clip_on=False)

        # Cosmetics: no title, no xticks, clean spines
        ax.set_ylabel(r"$\leftarrow$ Avg. Rank", fontsize=14, fontweight="bold")
        ax.set_xticks([])
        ax.set_xlim(xs[0] - 0.7, xs[-1] + 0.7)
        ax.set_ylim(0, label_tops.max() * 1.55 if rank_sem is not None
                       else ranks_shown.max() * 1.55)
        for side in ("top", "right", "bottom"):
            ax.spines[side].set_visible(False)
        ax.tick_params(axis="x", bottom=False)
        ax.tick_params(axis="y", labelsize=9)
        ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.6, zorder=1)
        ax.set_axisbelow(True)

        plt.tight_layout()
        fig.savefig(f"{RESULTS_DIR}/fig_paper_avg_rank_bars.pdf",
                    dpi=FIG_DPI, bbox_inches="tight")
        plt.show()


### 2.5.2  Per-Model Rank Scatter

Per-optimizer mean rank broken down by model. Each marker is one model; x-axis = optimizer (sorted best → worst by global mean rank); y-axis = mean rank on that model's tasks. Error bars show ±1 SEM across tasks.

In [ ]:
# Per-model scatter: x = optimizer (sorted by global mean LOSS rank),
# y = mean rank | mean METRIC (loss) | mean BLEU per model (toggle PLOT_MODE).
# Note: x-order is locked to loss-rank in every mode, so the three views are
# directly comparable (loss-winners on the left across modes).
# Each model gets a distinct marker + color; error bars = +/-1 SEM.
# Aligned with the heatmap above: same task_ranks / score_mat_raw,
# so all non-blacklisted optimizers appear by default.
# ----- Flags ---------------------------------------------------------------
import re

USE_PAPER_WHITELIST_SCATTER = False  # True restricts to PAPER_FIG_WHITELIST
SHOW_RANK_ERRORBARS = True           # toggle +/-1 SEM error bars
SHOW_AVG_MARKER     = True           # add a star for the global average
PLOT_MODE           = "rank"         # "rank" | "loss" (raw METRIC) | "bleu" (csv_ii)
Y_LOG_LOSS          = True           # log y-scale when PLOT_MODE == "loss"

# Sort legend by model parameter count (smallest -> largest). We pull the
# largest "<num>B" / "<num>b" token in the name (so "gemma-4-26B-A4B-it"
# sorts as 26B, not 4B-active).
def _model_size_b(name: str) -> float:
    matches = re.findall(r"(\d+(?:\.\d+)?)\s*[Bb](?![A-Za-z0-9_])", name)
    return max(float(m) for m in matches) if matches else float("inf")

def _pretty_model(name: str) -> str:
    short = name.split("/")[-1]
    short = short.replace("gemma", "Gemma")
    # Uppercase param-count "b" after a digit (e.g. "12b" -> "12B").
    short = re.sub(r"(\d)b", r"\1B", short)
    return short

if "score_mat_raw" not in dir() or "task_ranks" not in dir():
    print("Needs score_mat_raw / task_ranks from the Friedman + heatmap cells - skipped.")
elif "mean_ranks" not in dir():
    print("Needs mean_ranks from the Friedman cell - skipped.")
else:
    # ----- Optimizer ordering: global mean rank, best first ----------------
    if USE_PAPER_WHITELIST_SCATTER and "PAPER_FIG_WHITELIST" in dir():
        opt_order = [o for o in PAPER_FIG_WHITELIST if o in mean_ranks.index]
    else:
        opt_order = mean_ranks.index.tolist()

    # ----- Per-model statistics --------------------------------------------
    # PLOT_MODE picks the matrix being aggregated; opt order stays loss-rank-based.
    if PLOT_MODE == "loss":
        base_mat = score_mat_raw          # METRIC values from csv_i (e.g. best_loss)
    elif PLOT_MODE == "bleu":
        # Build a BLEU twin of score_mat_raw from csv_ii on the fly.
        # Same task-key + soft-exclude logic as the Friedman cell, so coverage
        # is comparable across modes. BLACKLIST_OPTIMIZERS already applied at
        # load time.
        if csv_ii.empty or "bleu" not in csv_ii.columns:
            raise RuntimeError("PLOT_MODE='bleu' but csv_ii has no 'bleu' column.")
        _df_b = csv_ii.copy()
        if "is_soft" in _df_b.columns:
            _df_b = _df_b[~_df_b["is_soft"].astype(bool)]
        _bl_keys = ["model_name", "msg_id"] + (["seed"] if TASK_INCLUDES_SEED else [])
        _bl_scores = (_df_b.groupby(["optimizer_name"] + _bl_keys)["bleu"]
                           .mean().reset_index())
        base_mat = _bl_scores.pivot_table(
            index=_bl_keys, columns="optimizer_name", values="bleu",
        )
    else:
        base_mat = task_ranks             # per-task integer ranks
    grouped      = base_mat.groupby(level="model_name")
    val_mean_df  = grouped.mean().T       # shape: opt x model
    val_std_df   = grouped.std(ddof=1).T
    val_n_df     = grouped.count().T
    val_sem_df   = val_std_df / np.sqrt(val_n_df)

    opt_order = sorted(
        [o for o in opt_order if o in val_mean_df.index],
        key=lambda o: mean_ranks.get(o, float("inf")),
    )
    # Legend order = small -> large model
    models = sorted(val_mean_df.columns, key=_model_size_b)

    # ----- Aesthetics ------------------------------------------------------
    # Okabe-Ito-inspired palette + distinct marker set
    _PALETTE  = ["#E63946", "#457B9D", "#2A9D8F", "#E9A020",
                 "#F4A261", "#8338EC", "#06D6A0"]
    _MARKERS  = ["o", "s", "^", "D", "P", "X", "v"]
    MODEL_COLORS  = _PALETTE[:len(models)]
    MODEL_MARKERS = _MARKERS[:len(models)]

    rc = {
        "font.family":      "serif",
        "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "axes.grid":        False,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(max(9, len(opt_order) * 0.78), 4.2))

        xs      = np.arange(len(opt_order))  # uniform spacing; order = mean rank
        n_m     = len(models)
        _jw     = 0.22
        jitters = np.linspace(-_jw, _jw, n_m) if n_m > 1 else np.array([0.0])

        for mi, (model, color, marker, jitter) in enumerate(
            zip(models, MODEL_COLORS, MODEL_MARKERS, jitters)
        ):
            ys   = val_mean_df.loc[opt_order, model].values
            errs = val_sem_df.loc[opt_order, model].values if SHOW_RANK_ERRORBARS else None
            ax.errorbar(
                xs + jitter, ys,
                yerr=errs,
                fmt=marker,
                color=color,
                label=_pretty_model(model),
                markersize=6,
                linewidth=0,
                elinewidth=0.9,
                capsize=3,
                capthick=0.9,
                markeredgecolor="white",
                markeredgewidth=0.7,
                alpha=0.75,
                zorder=3,
            )

        # ----- Axes & cosmetics -------------------------------------------
        ax.set_xticks(xs)
        ax.set_xticklabels(
            [optimizer_label(o) for o in opt_order],
            rotation=45, ha="right", fontsize=15,
        )
        if PLOT_MODE == "loss":
            _metric_pretty = METRIC.replace("_", " ").title()
            ax.set_ylabel(rf"Mean {_metric_pretty} ($\leftarrow$ better)", fontsize=17)
            if Y_LOG_LOSS:
                ax.set_yscale("log")
        elif PLOT_MODE == "bleu":
            # BLEU is higher=better -> arrow flips.
            ax.set_ylabel(r"Mean BLEU ($\rightarrow$ better)", fontsize=17)
        else:
            ax.set_ylabel(r"Mean Rank ($\leftarrow$ better)", fontsize=17)

        # Full black border (common ML-paper style; keeps all 4 spines).
        for side in ("top", "right", "left", "bottom"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(0.9)
            ax.spines[side].set_color("black")

        ax.tick_params(axis="y", labelsize=14, length=3)
        ax.tick_params(axis="x", which="major", length=4, width=0.8, color="#444444")
        ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.7, color="#666666", zorder=1)
        ax.set_axisbelow(True)
        ax.set_xlim(xs[0] - 0.65, xs[-1] + 0.65)

        # ----- Optional global-average marker -----------------------------
        if SHOW_AVG_MARKER:
            if PLOT_MODE == "rank":
                avg_ys = mean_ranks.loc[opt_order].values
            else:  # "loss" / "bleu" — average across tasks of the chosen matrix
                avg_ys = base_mat[opt_order].mean(axis=0).values
            ax.scatter(
                xs, avg_ys,
                marker="*", s=160, linewidths=1.0,
                color="#111111", zorder=5, label="Avg (all models)",
            )

        # Legend inside the plot, in a free corner (auto-picked).
        ax.legend(
            fontsize=13,
            loc="best",
            framealpha=0.9, edgecolor="#cccccc",
        )

        plt.tight_layout()
        fig.savefig(
            f"{RESULTS_DIR}/fig_per_model_{PLOT_MODE}_scatter.pdf",
            dpi=FIG_DPI, bbox_inches="tight",
        )
        plt.show()


### 2.5.3  Property Correlations (Mean Loss vs Mean Rank vs Mean BLEU)

One scalar per optimizer per property: `mean_loss` (avg of `best_loss` across tasks), `mean_rank` (Friedman mean rank), `mean_bleu` (avg BLEU across tasks). The correlation table shows how the three rankings agree; the scatter cell below plots any two of them head-to-head — pick the axes via `X_PROP` / `Y_PROP` at the top of the cell.


In [ ]:
# Per-optimizer property table: one row per optimizer, three columns.
# - mean_loss: avg METRIC value across all (model, msg, seed) tasks (csv_i)
# - mean_rank: Friedman mean rank from cell 2.5 (lower = better)
# - mean_bleu: avg BLEU across tasks (csv_ii)
# Drops optimizers missing any of the three (so the corr matrix is fair).
import pandas as pd

def _build_bleu_mat():
    """BLEU twin of score_mat_raw, built on the fly from csv_ii."""
    if csv_ii.empty or "bleu" not in csv_ii.columns:
        return None
    _df = csv_ii.copy()
    if "is_soft" in _df.columns:
        _df = _df[~_df["is_soft"].astype(bool)]
    _keys = ["model_name", "msg_id"] + (["seed"] if TASK_INCLUDES_SEED else [])
    _scores = (_df.groupby(["optimizer_name"] + _keys)["bleu"]
                  .mean().reset_index())
    return _scores.pivot_table(index=_keys, columns="optimizer_name", values="bleu")

bleu_mat_raw = _build_bleu_mat()

prop_df = pd.DataFrame({
    "mean_loss": score_mat_raw.mean(axis=0),
    "mean_rank": mean_ranks,
    "mean_bleu": (bleu_mat_raw.mean(axis=0)
                  if bleu_mat_raw is not None else pd.Series(dtype=float)),
})
prop_df = prop_df.dropna(how="any").sort_values("mean_rank")
print(f"Optimizers with all 3 properties: {len(prop_df)}")
print("\nPer-optimizer summary:")
print(prop_df.round(3).to_string())

print("\nPearson correlation (3x3):")
print(prop_df.corr(method="pearson").round(3).to_string())

print("\nSpearman correlation (3x3):")
print(prop_df.corr(method="spearman").round(3).to_string())


In [ ]:
# ----- Pick the two properties to compare ---------------------------------
X_PROP = "mean_rank"   # one of: "mean_loss" | "mean_rank" | "mean_bleu"
Y_PROP = "mean_bleu"

ANNOTATE_POINTS = True             # write optimizer label next to each dot
SHOW_LINEAR_FIT = True             # dashed least-squares line
X_LOG = (X_PROP == "mean_loss")    # losses span many decades; auto-log them
Y_LOG = (Y_PROP == "mean_loss")
# --------------------------------------------------------------------------

# Higher-is-better arrow for BLEU, lower-is-better arrow otherwise.
_DIR = {"mean_loss": r"$\leftarrow$", "mean_rank": r"$\leftarrow$",
        "mean_bleu": r"$\rightarrow$"}
_PRETTY = {"mean_loss": f"Mean {METRIC.replace('_',' ').title()}",
           "mean_rank": "Mean Rank",
           "mean_bleu": "Mean BLEU"}

if "prop_df" not in dir():
    print("Run the property-table cell first - skipped.")
elif X_PROP == Y_PROP:
    print(f"X_PROP and Y_PROP are both {X_PROP!r} - pick two different properties.")
else:
    sub = prop_df[[X_PROP, Y_PROP]].dropna()
    xs = sub[X_PROP].values
    ys = sub[Y_PROP].values

    pearson  = sub.corr(method="pearson").iloc[0, 1]
    spearman = sub.corr(method="spearman").iloc[0, 1]

    rc = {
        "font.family":      "serif",
        "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "axes.grid":        False,
    }

    with plt.rc_context(rc):
        fig, ax = plt.subplots(figsize=(6.5, 5.0))

        for opt in sub.index:
            ax.scatter(
                sub.loc[opt, X_PROP], sub.loc[opt, Y_PROP],
                color=optimizer_color(opt), s=110,
                edgecolor="black", linewidth=0.6, zorder=3,
            )
            if ANNOTATE_POINTS:
                ax.annotate(
                    optimizer_label(opt),
                    (sub.loc[opt, X_PROP], sub.loc[opt, Y_PROP]),
                    textcoords="offset points", xytext=(5, 4),
                    fontsize=10, color="#222222",
                )

        # Linear fit (visual only). Skip when an axis is log -- a line in
        # log space is misleading without log-fitting the data.
        if SHOW_LINEAR_FIT and len(sub) >= 2 and not (X_LOG or Y_LOG):
            m, b = np.polyfit(xs, ys, 1)
            xx = np.linspace(xs.min(), xs.max(), 50)
            ax.plot(xx, m * xx + b, linestyle="--", color="#666666",
                    linewidth=1.0, zorder=1)

        if X_LOG: ax.set_xscale("log")
        if Y_LOG: ax.set_yscale("log")

        ax.set_xlabel(f"{_PRETTY[X_PROP]} ({_DIR[X_PROP]} better)", fontsize=14)
        ax.set_ylabel(f"{_PRETTY[Y_PROP]} ({_DIR[Y_PROP]} better)", fontsize=14)
        ax.set_title(
            f"Pearson r = {pearson:+.2f}    Spearman r = {spearman:+.2f}    "
            f"(n = {len(sub)} optimizers)",
            fontsize=11,
        )

        # Full black border (matches 2.5.2).
        for side in ("top", "right", "left", "bottom"):
            ax.spines[side].set_visible(True)
            ax.spines[side].set_linewidth(0.9)
            ax.spines[side].set_color("black")

        ax.tick_params(axis="both", labelsize=12, length=4, width=0.8, color="#444444")
        ax.grid(linestyle=":", linewidth=0.8, alpha=0.7, color="#666666", zorder=1)
        ax.set_axisbelow(True)

        plt.tight_layout()
        fig.savefig(
            f"{RESULTS_DIR}/fig_prop_scatter_{X_PROP}_vs_{Y_PROP}.pdf",
            dpi=FIG_DPI, bbox_inches="tight",
        )
        plt.show()


## 3  Average BLEU per Optimizer (CSV II)

In [ ]:
agg_bleu = bar_plot(csv_ii, "bleu", "Average BLEU Score per Optimizer", "BLEU (↑ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_bleu.pdf")
agg_bleu


## 4  Average Universality per Optimizer (CSV III)

Universality = mean `strongreject_finetuned` score across ClearHarm messages.

In [ ]:
agg_univ = bar_plot(csv_iii, "strongreject_finetuned",
                    "Average Universality Score per Optimizer",
                    "Universality / Jailbreakness (↑ = more harmful)",
                    output_file=f"{RESULTS_DIR}/fig_avg_universality.pdf")
agg_univ


## 5  Box Plots

First average across seeds per (optimizer, model, msg_id) to reduce seed noise, then plot distribution across messages/models.

In [ ]:
# Filter control — set to None to include all models
FILTER_MODELS = None  # e.g. ["google/gemma-2-2b-it"]

def box_plot(df, value_col, title, ylabel, output_file=None,
             exclude_soft=True):
    plot_df = df.copy()
    if exclude_soft and "is_soft" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_soft"].astype(bool)]
    if FILTER_MODELS:
        plot_df = plot_df[plot_df["model_name"].isin(FILTER_MODELS)]

    # Average across seeds first
    seed_avg = (plot_df.groupby(["optimizer_name", "model_name", "msg_id"])[value_col]
                .mean().reset_index())

    fig, ax = plt.subplots(figsize=(max(8, seed_avg["optimizer_name"].nunique() * 0.9), 5))
    order = (seed_avg.groupby("optimizer_name")[value_col].median()
             .sort_values().index.tolist())
    sns.boxplot(data=seed_avg, x="optimizer_name", y=value_col, order=order,
                palette={o: optimizer_color(o) for o in order}, ax=ax, width=0.6)
    ax.set_xticklabels([optimizer_label(o) for o in order])
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

box_plot(csv_i,   "best_loss",              "Loss Distribution per Optimizer",         "Loss (↓ better)",
         f"{RESULTS_DIR}/fig_box_loss.pdf")
box_plot(csv_ii,  "bleu",                   "BLEU Distribution per Optimizer",          "BLEU (↑ better)",
         f"{RESULTS_DIR}/fig_box_bleu.pdf")
box_plot(csv_iii, "strongreject_finetuned", "Universality Distribution per Optimizer", "Universality (↑ = more harmful)",
         f"{RESULTS_DIR}/fig_box_universality.pdf")

## 6  LaTeX Table

Rows = optimizers, columns = models, cell = avg across seeds & instructions. Winners per column are marked in **bold**.

In [ ]:
METRIC_COL = "best_loss"  # swap to "bleu" or "strongreject_finetuned" for other metrics
METRIC_LOWER_IS_BETTER = True  # set False for BLEU / universality

df_tab = csv_i.copy()
if "is_soft" in df_tab.columns:
    df_tab = df_tab[~df_tab["is_soft"].astype(bool)]

# Average across seeds and msg_ids per (optimizer, model)
pivot = (df_tab.groupby(["optimizer_name", "model_name"])[METRIC_COL]
         .mean().unstack("model_name"))

# Add overall average column
pivot["Avg"] = pivot.mean(axis=1)
pivot = pivot.sort_values("Avg", ascending=METRIC_LOWER_IS_BETTER)

# Build LaTeX
model_cols = [c for c in pivot.columns if c != "Avg"]
header_row = " & ".join(["Optimizer"] + [c.split("/")[-1] for c in model_cols] + ["Avg"]) + r" \\"

rows_latex = []
for opt, row in pivot.iterrows():
    vals = [row[c] for c in model_cols] + [row["Avg"]]
    # Find winner (min or max) per column
    formatted = []
    for col_idx, (col, v) in enumerate(zip(model_cols + ["Avg"], vals)):
        col_vals = pivot[col].dropna()
        if pd.isna(v):
            formatted.append("—")
            continue
        is_winner = (v == col_vals.min()) if METRIC_LOWER_IS_BETTER else (v == col_vals.max())
        cell = f"\\textbf{{{v:.3f}}}" if is_winner else f"{v:.3f}"
        formatted.append(cell)
    rows_latex.append(f"    {opt} & " + " & ".join(formatted) + r" \\")

print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Optimizer Benchmark — " + METRIC_COL + "}")
print(r"\begin{tabular}{l" + "r" * (len(model_cols) + 1) + "}")
print(r"\toprule")
print(f"    {header_row}")
print(r"\midrule")
for r in rows_latex:
    print(r)
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

---
# Exp2: Jailbreak Tweaks Analysis

Load exp2 CSVs (single and multi-instruction). Group by `variant_name` instead of `optimizer_name`.

**Controls**: Set `EXP2_RESULTS_DIR` and CSV filenames below to match your `run_all.sh` output.

In [ ]:
# ── Exp2 Configuration ──────────────────────────────────────────────
EXP2_RESULTS_DIR = RESULTS_DIR  # same dir by default

# exp2 is pinned to a single model (see run_all.sh EXP2_MODELS).
EXP2_MODEL_SLUG = "gemma-3-12b-it"

# Single-instruction CSVs
EXP2_SINGLE_CSV_I   = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_i.csv"
EXP2_SINGLE_CSV_II  = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_ii.csv"
EXP2_SINGLE_CSV_III = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_iii.csv"

# Multi-instruction CSVs
EXP2_MULTI_CSV_I   = f"{EXP2_RESULTS_DIR}/exp2_multi_{EXP2_MODEL_SLUG}_csv_i.csv"
EXP2_MULTI_CSV_III = f"{EXP2_RESULTS_DIR}/exp2_multi_{EXP2_MODEL_SLUG}_csv_iii.csv"

# Load
import os
exp2s_i   = pd.read_csv(EXP2_SINGLE_CSV_I)   if os.path.exists(EXP2_SINGLE_CSV_I)   else pd.DataFrame()
exp2s_ii  = pd.read_csv(EXP2_SINGLE_CSV_II)  if os.path.exists(EXP2_SINGLE_CSV_II)  else pd.DataFrame()
exp2s_iii = pd.read_csv(EXP2_SINGLE_CSV_III) if os.path.exists(EXP2_SINGLE_CSV_III) else pd.DataFrame()
exp2m_i   = pd.read_csv(EXP2_MULTI_CSV_I)    if os.path.exists(EXP2_MULTI_CSV_I)    else pd.DataFrame()
exp2m_iii = pd.read_csv(EXP2_MULTI_CSV_III)   if os.path.exists(EXP2_MULTI_CSV_III)  else pd.DataFrame()

for name, df in [("exp2 single I", exp2s_i), ("exp2 single II", exp2s_ii),
                  ("exp2 single III", exp2s_iii), ("exp2 multi I", exp2m_i),
                  ("exp2 multi III", exp2m_iii)]:
    print(f"{name}: {len(df)} rows")

## Exp2 Single-Instruction: Loss, BLEU, Universality

Same plots as exp1 but grouped by `variant_name`.

In [ ]:
GROUP_COL = "variant_name"  # exp2 groups by variant, not optimizer

def bar_plot_exp2(df, value_col, title, ylabel, group_col=GROUP_COL,
                  output_file=None, yscale="linear"):
    if df.empty:
        print(f"No data for: {title}")
        return
    agg = (df.groupby(group_col)[value_col]
           .agg(["mean", "std", "count"]).reset_index().sort_values("mean"))
    agg["se"] = agg["std"] / np.sqrt(agg["count"])
    fig, ax = plt.subplots(figsize=(max(8, len(agg) * 0.9), 5))
    colors = sns.color_palette(PALETTE, len(agg))
    ax.bar(agg[group_col], agg["mean"], yerr=agg["se"], capsize=4,
           color=colors, edgecolor="white")
    ax.set_xlabel("Variant")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if yscale != "linear":
        # symlog tolerates the negative losses some variants produce
        # (e.g. gcg_attn_hijack, gcg_flrt_clamp w/ clamp_min_nll).
        ax.set_yscale(yscale)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        print(f"Saving figure to {output_file}")
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return agg

# Loss uses symlog: variants combine different losses (steering ~1e8, hijack
# ~ -1, vanilla ~0..2), so a linear scale is uninformative. The CSV's
# `best_loss` is the *combined* objective, not directly comparable across
# variants — for a fair comparison see the dynamics plot below
# (loss/PrefillCELoss).
bar_plot_exp2(exp2s_i,   "best_loss",              "Exp2 Single: Avg Loss per Variant (symlog)", "Loss", yscale="symlog")
bar_plot_exp2(exp2s_ii,  "bleu",                   "Exp2 Single: Avg BLEU per Variant",          "BLEU")
bar_plot_exp2(exp2s_iii, "strongreject_finetuned", "Exp2 Single: Avg Universality per Variant",  "Universality")


## Exp2 Loss Bars: Combined `best_loss` vs PrefillCE-only (Single-Instruction)

For each variant, bars the CSV's combined `best_loss` *side-by-side* with
the PrefillCE component (pulled from wandb's `loss/PrefillCELoss` metric).
The PrefillCE column lets you compare variants on a shared scale even when
their training objective is a combined loss.


In [ ]:
# # Pull final-best PrefillCE-only loss per variant from wandb, then plot
# # side-by-side with the CSV `best_loss`.
# #
# # We fetch runs by `run_id` from the CSV (rather than via api.runs(filters=...)),
# # because the server-side filter on `config.run_type=enhancebench_single` was
# # returning 0 runs — likely the runs never moved into state="finished".
# EXP2_PRES_KEY = "loss/PrefillCELoss"

# if not exp2s_i.empty:
#     api = wandb.Api()
#     csv_run_ids = exp2s_i["run_id"].dropna().unique().tolist()
#     print(f"CSV has {len(csv_run_ids)} unique run_ids")

#     rows = []
#     skipped_no_key = skipped_load_err = 0
#     for rid in csv_run_ids:
#         try:
#             run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{rid}")
#         except Exception:
#             skipped_load_err += 1
#             continue
#         cfg = run.config
#         var = cfg.get("variant_name", "unknown")
#         seed = cfg.get("seed", 0)
#         msg_id = cfg.get("msg_id", None)
#         try:
#             hist = run.history(keys=[EXP2_PRES_KEY], x_axis="_step", pandas=True)
#         except Exception:
#             hist = pd.DataFrame()
#         if hist.empty or EXP2_PRES_KEY not in hist.columns:
#             skipped_no_key += 1
#             continue
#         vals = hist[EXP2_PRES_KEY].dropna()
#         if vals.empty:
#             skipped_no_key += 1
#             continue
#         rows.append({"variant_name": var, "seed": seed, "msg_id": msg_id,
#                      "best_prefillce": float(vals.min())})
#     if skipped_load_err:
#         print(f"  skipped {skipped_load_err} runs that failed to load")
#     if skipped_no_key:
#         print(f"  skipped {skipped_no_key} runs missing `{EXP2_PRES_KEY}` in history")
#     prefillce_df = pd.DataFrame(rows)
#     n_var = prefillce_df["variant_name"].nunique() if not prefillce_df.empty else 0
#     print(f"kept {len(prefillce_df)} runs across {n_var} variants")

#     # Side-by-side grouped bar: combined best_loss (CSV) vs PrefillCE-only (wandb).
#     if not prefillce_df.empty:
#         loss_long = (exp2s_i[["variant_name", "best_loss"]]
#                      .rename(columns={"best_loss": "value"})
#                      .assign(metric="best_loss (combined)"))
#         pce_long = (prefillce_df[["variant_name", "best_prefillce"]]
#                     .rename(columns={"best_prefillce": "value"})
#                     .assign(metric="loss/PrefillCELoss"))
#         long = pd.concat([loss_long, pce_long], ignore_index=True)
#         order = (long.groupby("variant_name")["value"].mean()
#                  .sort_values().index.tolist())
#         fig, ax = plt.subplots(figsize=(max(8, len(order) * 1.1), 5))
#         sns.barplot(data=long, x="variant_name", y="value", hue="metric",
#                     order=order, errorbar="se", ax=ax)
#         ax.set_xlabel("Variant")
#         ax.set_ylabel("Loss")
#         ax.set_title("Exp2 Single: best_loss (combined) vs PrefillCE per Variant (symlog)")
#         ax.set_yscale("symlog")  # negative + huge magnitudes coexist
#         plt.xticks(rotation=35, ha="right")
#         ax.legend(loc="best", fontsize=9)
#         plt.tight_layout()
#         plt.show()
#     else:
#         print("No PrefillCE data; falling back to combined-loss bar only.")
#         bar_plot_exp2(exp2s_i, "best_loss",
#                       "Exp2 Single: Avg Loss per Variant (symlog)",
#                       "Loss", yscale="symlog")
# else:
#     print("No exp2 single-instruction data available.")


## Exp2 Box Plots (Single-Instruction)

Per-suffix distribution across messages, models, and seeds (no averaging).


In [ ]:
def box_plot_exp2(df, value_col, title, ylabel, group_col=GROUP_COL,
                  output_file=None, yscale="linear", per_suffix=False):
    if df.empty:
        print(f"No data for: {title}")
        return
    if per_suffix:
        # One point per suffix: aggregate over msg_id, keeping (variant, model, seed, trigger_id).
        suffix_keys = [k for k in [group_col, "model_name", "seed", "trigger_id"] if k in df.columns]
        data = (df.groupby(suffix_keys)[value_col].mean().reset_index())
    else:
        # One point per row (per suffix-message pair).
        data = df[[group_col, value_col]].dropna()
    order = (data.groupby(group_col)[value_col].mean()
             .sort_values().index.tolist())
    fig, ax = plt.subplots(figsize=(max(8, data[group_col].nunique() * 0.9), 5))
    sns.boxplot(data=data, x=group_col, y=value_col, order=order,
                palette=PALETTE, ax=ax, width=0.6, showfliers=False)
    sns.stripplot(data=data, x=group_col, y=value_col, order=order,
                  color="black", size=2.5, alpha=0.4, jitter=0.25, ax=ax)
    ax.set_xticklabels([variant_label(t.get_text()) for t in ax.get_xticklabels()])
    ax.set_xlabel("Variant")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if yscale != "linear":
        ax.set_yscale(yscale)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

box_plot_exp2(exp2s_i,   "best_loss",              "Exp2: Loss Distribution per Variant (symlog)", "Loss", yscale="symlog")
box_plot_exp2(exp2s_ii,  "bleu",                   "Exp2: BLEU Distribution per Variant",          "BLEU")
box_plot_exp2(exp2s_iii, "strongreject_finetuned", "Exp2: Universality Distribution per Variant",  "Universality", per_suffix=True)


### Universality Box Plot — Paper-Ready

Same per-suffix aggregation as the cell above, restyled for the paper: Times serif font, larger labels, horizontal layout, no y-axis, no title. Top row is "Base" (gcg_vanilla); the remaining variants follow from worst-to-best mean universality, top to bottom (best non-Base variant at the bottom). The resulting variant order is exported as `PAPER_VARIANT_ORDER` so the multi-instruction paper-ready plot can reuse it.

In [ ]:
# Paper-ready Universality box plot (same data as box_plot_exp2 per_suffix=True).
SHOW_POINTS = True  # overlay individual suffixes as subtle dots on the boxes

plt.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize":   20,
    "axes.labelsize":   20,
    "xtick.labelsize":  17,
    "ytick.labelsize":  17,
    "legend.fontsize":  16,
})

df        = exp2s_iii
value_col = "strongreject_finetuned"

# Per-suffix aggregation (variant, model, seed, trigger_id) over msg_id.
suffix_keys = [k for k in [GROUP_COL, "model_name", "seed", "trigger_id"] if k in df.columns]
data  = df.groupby(suffix_keys)[value_col].mean().reset_index()

# Top row = "Base" (gcg_vanilla); the rest follow from worst-to-best mean,
# top to bottom (so worst non-Base variant sits right under Base and the
# best non-Base variant ends up at the bottom). In seaborn horizontal
# cat plots `order[0]` renders at the top, so we sort ascending by mean.
all_means  = data.groupby(GROUP_COL)[value_col].mean()
non_base   = all_means.drop("gcg_vanilla", errors="ignore")
rest       = non_base.sort_values(ascending=True).index.tolist()
order      = (["gcg_vanilla"] + rest) if "gcg_vanilla" in all_means.index else rest
# Share this order with the multi-instruction paper-ready plot below.
PAPER_VARIANT_ORDER = order
# Sequential palette (worst -> best): low-saturation blues -> teal.
palette = sns.color_palette("crest", n_colors=len(order))

fig, ax = plt.subplots(figsize=(8, max(3.8, 0.6 * len(order))))
sns.boxplot(data=data, y=GROUP_COL, x=value_col, order=order,
            palette=palette, ax=ax, width=0.65, showfliers=False, linewidth=1.4)
if SHOW_POINTS:
    # Subtle but visible: small dark-grey dots, low alpha, light edge for separation.
    sns.stripplot(data=data, y=GROUP_COL, x=value_col, order=order,
                  color="#2b2b2b", size=2.6, alpha=0.4, jitter=0.22,
                  linewidth=0.4, edgecolor="white", ax=ax)

ax.set_yticklabels([variant_label(t.get_text()) for t in ax.get_yticklabels()])
ax.set_xlabel(r"Universality ($\rightarrow$ better)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("")
ax.set_title("")

ax.tick_params(axis="y", length=0)
# Black frame around the plot.
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("black")
    spine.set_linewidth(1.0)
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", visible=False)

plt.tight_layout()
out_path = f"{RESULTS_DIR}/fig_exp2_universality_box_paper.pdf"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


### Exp2 Single-Instruction: Final-Loss Stats & Winrate

Per-variant mean / std of `best_loss` (the final best after the 500-step budget) plus per-task winrate. A task is one `(model_name, msg_id, seed)` tuple; the variant with the lowest `best_loss` on that task gets the win (ties split evenly across the tied variants).

In [ ]:
# Per-variant final-loss stats and winrate over Exp2 single-instruction runs.
# Final loss = `best_loss` at the end of the 500-step budget.
# Winrate    = fraction of (model, msg, seed) tasks where the variant has the
#              lowest best_loss (ties split evenly across tied variants).
if exp2s_i.empty:
    print("No exp2 single-instruction data available.")
else:
    task_keys = [k for k in ["model_name", "msg_id", "seed"] if k in exp2s_i.columns]
    # Collapse any duplicate runs per (task, variant) before pivoting.
    per_task = (exp2s_i.groupby(task_keys + ["variant_name"])["best_loss"]
                .mean().reset_index())
    wide = per_task.pivot_table(index=task_keys, columns="variant_name",
                                 values="best_loss")

    # Per-task wins: 1/k split across the k variants tied for the min.
    min_per_task = wide.min(axis=1)
    is_winner = wide.eq(min_per_task, axis=0)
    n_winners = is_winner.sum(axis=1).replace(0, np.nan)
    win_share = is_winner.div(n_winners, axis=0)
    n_tasks_per_variant = wide.notna().sum(axis=0)
    winrate = win_share.sum(axis=0) / n_tasks_per_variant

    stats = (exp2s_i.groupby("variant_name")["best_loss"]
             .agg(["mean", "std", "count"])
             .rename(columns={"count": "n_runs"}))
    stats["winrate"]    = winrate
    stats["n_tasks"]    = n_tasks_per_variant
    stats = stats.sort_values("mean")

    # Pretty print: lowest mean loss first; appended Base row stays last.
    if "gcg_vanilla" in stats.index:
        ordered = [v for v in stats.index if v != "gcg_vanilla"] + ["gcg_vanilla"]
        stats = stats.loc[ordered]

    label_w = max(len(variant_label(v)) for v in stats.index)
    n_total_tasks = len(wide)
    print(f"Exp2 single-instruction: {n_total_tasks} tasks "
          f"across {len(stats)} variants "
          f"(task = ({', '.join(task_keys)}))")
    print()
    print(f"  {'Variant':<{label_w}}   mean      std       winrate   n_runs  n_tasks")
    for var, row in stats.iterrows():
        print(f"  {variant_label(var):<{label_w}}   "
              f"{row['mean']:>+8.4f}  {row['std']:>7.4f}   "
              f"{row['winrate']:>6.1%}   "
              f"{int(row['n_runs']):>5}   {int(row['n_tasks']):>5}")


## Exp2 Multi-Instruction: Loss & Universality

Multi-instruction runs produce a single universal trigger per variant. 
CSV III universality is the primary metric here.

In [ ]:
bar_plot_exp2(exp2m_i,   "best_loss",              "Exp2 Multi: Avg Loss per Variant (symlog)", "Loss", yscale="symlog")
bar_plot_exp2(exp2m_iii, "strongreject_finetuned", "Exp2 Multi: Avg Universality per Variant",  "Universality")


### Universality Plot — Paper-Ready (Multi-Instruction)

Same paper styling as the single-instruction box plot above. With only ~3 runs per variant we drop the box and show enlarged dots plus a short vertical mean tick per variant.

In [ ]:
# Paper-ready Universality plot -- multi-instruction (~3 runs per variant).
# No boxes: bigger dots, plus a short vertical line marking the mean per variant.
SHOW_POINTS = True

plt.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize":   20,
    "axes.labelsize":   20,
    "xtick.labelsize":  17,
    "ytick.labelsize":  17,
    "legend.fontsize":  16,
})

df        = exp2m_iii
value_col = "strongreject_finetuned"

# One row per multi run (variant, model, seed, trigger_id).
keys = [k for k in [GROUP_COL, "model_name", "seed", "trigger_id"] if k in df.columns]
data = df.groupby(keys)[value_col].mean().reset_index()

# Reuse the order pinned by the single-instruction paper-ready plot above
# (Base on top, the rest sorted by single-instruction mean). Drop variants
# that aren't present in the multi-instruction data.
present = set(data[GROUP_COL].unique())
order   = [v for v in PAPER_VARIANT_ORDER if v in present]
palette = sns.color_palette("crest", n_colors=len(order))

fig, ax = plt.subplots(figsize=(8, max(3.8, 0.6 * len(order))))

# Per-variant vertical mean tick, colored to match the palette.
means = data.groupby(GROUP_COL)[value_col].mean()
for i, variant in enumerate(order):
    ax.vlines(x=means[variant], ymin=i - 0.28, ymax=i + 0.28,
              color=palette[i], linewidth=2.8, zorder=2)

if SHOW_POINTS:
    # Bigger and slightly more present than the single-instruction stripplot,
    # since we only have a handful of points per row.
    sns.stripplot(data=data, y=GROUP_COL, x=value_col, order=order,
                  color="#2b2b2b", size=6.0, alpha=0.6, jitter=0.18,
                  linewidth=0.5, edgecolor="white", ax=ax, zorder=3)

ax.set_yticklabels([variant_label(t.get_text()) for t in ax.get_yticklabels()])
ax.set_xlabel(r"Universality ($\rightarrow$ better)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("")
ax.set_title("")
ax.tick_params(axis="y", length=0)
# Black frame around the plot.
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("black")
    spine.set_linewidth(1.0)
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", visible=False)

plt.tight_layout()
out_path = f"{RESULTS_DIR}/fig_exp2_multi_universality_paper.pdf"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


### Universality Box+Multi — Paper-Ready (Combined)

Same paper styling as the single- and multi-instruction plots above, merged into one figure: the per-suffix box plot from the single-instruction runs is rendered on a wider canvas, with the ~3 multi-instruction runs per variant overlaid as a distinct marker and the multi-instruction per-variant mean shown with its own marker. The row order is reused from `PAPER_VARIANT_ORDER` (Base on top; rest from worst to best by single-instruction mean, top to bottom).

In [ ]:
# Combined paper-ready Universality plot:
#   - boxes      = single-instruction per-suffix distribution (same as the
#                  single-instruction paper plot above)
#   - diamonds   = individual multi-instruction runs (~3 per variant)
#   - star       = per-variant mean of the multi-instruction runs
# Wider canvas (so the ~3 multi points stay distinguishable inside each row).
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize":   20,
    "axes.labelsize":   20,
    "xtick.labelsize":  17,
    "ytick.labelsize":  17,
    "legend.fontsize":  15,
})

value_col = "strongreject_finetuned"

# Single-instruction: one point per suffix (same agg as the single paper plot).
suffix_keys = [k for k in [GROUP_COL, "model_name", "seed", "trigger_id"] if k in exp2s_iii.columns]
data_single = exp2s_iii.groupby(suffix_keys)[value_col].mean().reset_index()

# Multi-instruction: one point per run (same agg as the multi paper plot).
multi_keys  = [k for k in [GROUP_COL, "model_name", "seed", "trigger_id"] if k in exp2m_iii.columns]
data_multi  = exp2m_iii.groupby(multi_keys)[value_col].mean().reset_index()

# Reuse the order pinned by the single-instruction paper-ready plot. Variants
# are kept only if they appear in the single data (since the boxes drive the
# main layout); multi points for missing variants are dropped silently.
present = set(data_single[GROUP_COL].unique())
order   = [v for v in PAPER_VARIANT_ORDER if v in present]
palette = sns.color_palette("crest", n_colors=len(order))

MULTI_COLOR = "#D55E00"  # warm orange, distinct from the cool crest boxes

# Wider figure than the per-source plots so the multi diamonds stay readable.
fig, ax = plt.subplots(figsize=(13, max(3.8, 0.6 * len(order))))

# Boxes (single-instruction).
sns.boxplot(data=data_single, y=GROUP_COL, x=value_col, order=order,
            palette=palette, ax=ax, width=0.65, showfliers=False, linewidth=1.4)

# Subtle per-suffix dots on the boxes (matches the single paper plot styling).
sns.stripplot(data=data_single, y=GROUP_COL, x=value_col, order=order,
              color="#2b2b2b", size=2.6, alpha=0.4, jitter=0.22,
              linewidth=0.4, edgecolor="white", ax=ax, zorder=3)

# Multi-instruction individual runs: distinct diamond marker, no jitter so
# they sit on the row's center line (only ~3 per variant, easy to read).
multi_subset = data_multi[data_multi[GROUP_COL].isin(order)]
sns.stripplot(data=multi_subset, y=GROUP_COL, x=value_col, order=order,
              marker="D", color=MULTI_COLOR, size=7.5, alpha=0.85, jitter=0.0,
              linewidth=0.7, edgecolor="white", ax=ax, zorder=4)

# Multi-instruction per-variant mean: hollow star, drawn on top.
multi_means = data_multi.groupby(GROUP_COL)[value_col].mean()
for i, variant in enumerate(order):
    if variant in multi_means.index:
        ax.scatter(multi_means[variant], i, marker="*", s=260,
                   facecolor="white", edgecolor=MULTI_COLOR, linewidths=1.6,
                   zorder=5)

# Legend (proxy artists -- one row per data source / aggregation).
legend_handles = [
    Patch(facecolor=palette[len(palette) // 2], edgecolor="black",
          label="Single-instruction (per suffix)"),
    Line2D([0], [0], marker="D", linestyle="", color=MULTI_COLOR,
           markersize=8, markeredgecolor="white", markeredgewidth=0.7,
           label="Multi-instruction run"),
    Line2D([0], [0], marker="*", linestyle="", markerfacecolor="white",
           markeredgecolor=MULTI_COLOR, markeredgewidth=1.6, markersize=15,
           label="Multi-instruction mean"),
]
ax.legend(handles=legend_handles, loc="lower right", frameon=True,
          framealpha=0.95, edgecolor="0.75", fancybox=False, borderpad=0.6,
          handlelength=1.6, handletextpad=0.6)

ax.set_yticklabels([variant_label(t.get_text()) for t in ax.get_yticklabels()])
ax.set_xlabel(r"Universality ($\rightarrow$ better)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.set_ylabel("")
ax.set_title("")

ax.tick_params(axis="y", length=0)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("black")
    spine.set_linewidth(1.0)
ax.grid(axis="x", linestyle="--", alpha=0.5)
ax.grid(axis="y", visible=False)

plt.tight_layout()
out_path = f"{RESULTS_DIR}/fig_exp2_universality_combined_paper.pdf"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()


## Exp2 Optimization Dynamics (Single-Instruction)

Same as exp1 dynamics plot but filtered by `tweakbench_single` run type. Lines = variants.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
EXP2_CHOSEN_MSG_ID = 0
EXP2_CHOSEN_MODEL  = exp2s_i["model_name"].iloc[0] if not exp2s_i.empty else "N/A"
EXP2_X_AXIS = "step"  # "step" or "flops"
# Use the PrefillCE component for fair comparison: variants combine PrefillCE
# with different secondary losses (steering, hijack, distill...) at different
# weights, so the aggregated `best_loss` isn't comparable across variants.
EXP2_Y_AXIS = "loss/PrefillCELoss"

print(f"Available msg_ids: {sorted(exp2s_i['msg_id'].dropna().unique().tolist()) if not exp2s_i.empty else []}")
print(f"Available models:  {exp2s_i['model_name'].unique().tolist() if not exp2s_i.empty else []}")

if not exp2s_i.empty:
    api = wandb.Api()
    # Fetch runs by run_id from the CSV. The server-side filter on
    # config.run_type was returning 0 runs (likely runs not in state="finished").
    sub = exp2s_i[(exp2s_i["model_name"] == EXP2_CHOSEN_MODEL)
                  & (exp2s_i["msg_id"] == EXP2_CHOSEN_MSG_ID)]
    csv_run_ids = sub["run_id"].dropna().unique().tolist()
    print(f"CSV rows for chosen model+msg: {len(sub)} ({len(csv_run_ids)} unique run_ids)")

    histories = {}
    skipped_load = skipped_empty = 0
    first_err = None
    for rid in csv_run_ids:
        try:
            run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{rid}")
        except Exception as e:
            skipped_load += 1
            if first_err is None:
                first_err = (rid, type(e).__name__, str(e)[:300])
            continue
        cfg = run.config
        var  = cfg.get("variant_name", "unknown")
        seed = cfg.get("seed", 0)
        try:
            hist = run.history(
                keys=[EXP2_Y_AXIS, "loss", "best_loss", "total_models_stats/total_flops"],
                x_axis="_step", pandas=True,
            )
        except Exception:
            hist = run.history(pandas=True)
        if hist.empty or EXP2_Y_AXIS not in hist.columns:
            skipped_empty += 1
            continue
        if "step" not in hist.columns and "_step" in hist.columns:
            hist = hist.rename(columns={"_step": "step"})
        if "total_models_stats/total_flops" in hist.columns:
            hist = hist.rename(columns={"total_models_stats/total_flops": "flops"})
        histories.setdefault(var, {})[seed] = hist

    print(f"Loaded {sum(len(v) for v in histories.values())} runs for {len(histories)} variants"
          f" (skipped {skipped_load} load errors, {skipped_empty} missing `{EXP2_Y_AXIS}`)")
    if first_err is not None:
        rid, ename, emsg = first_err
        print(f"  first load error: run_id={rid}: {ename}: {emsg}")
        print(f"  (path tried: {WANDB_ENTITY}/{WANDB_PROJECT}/{rid})")

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette(PALETTE, max(len(histories), 1))
    for color, (var_name, seed_hists) in zip(colors, histories.items()):
        x_col = "flops" if (EXP2_X_AXIS == "flops" and "flops" in next(iter(seed_hists.values())).columns) else "step"
        all_x = sorted(set(x for h in seed_hists.values() for x in h[x_col].dropna()))
        interp_losses = []
        for hist in seed_hists.values():
            valid = hist[[x_col, EXP2_Y_AXIS]].dropna()
            if len(valid) < 2:
                continue
            interp_losses.append(np.interp(all_x, valid[x_col], valid[EXP2_Y_AXIS]))
        if not interp_losses:
            continue
        arr = np.array(interp_losses)
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        ax.plot(all_x, mean, label=var_name, color=color)
        ax.fill_between(all_x, mean - std, mean + std, alpha=0.15, color=color)
    ax.set_xlabel("FLOPs" if EXP2_X_AXIS == "flops" else "Step")
    ax.set_ylabel(EXP2_Y_AXIS)
    # symlog handles the negative PrefillCE values produced by FLRT clamp.
    ax.set_yscale("symlog")
    ax.set_title(f"Exp2 Dynamics — {EXP2_CHOSEN_MODEL.split('/')[-1]}, msg {EXP2_CHOSEN_MSG_ID}")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No exp2 single-instruction data available.")


## Exp2 LaTeX Table (Single-Instruction)

Rows = variants, columns = models, cell = avg across seeds & instructions.

In [ ]:
EXP2_METRIC_COL = "best_loss"  # swap to "bleu" or "strongreject_finetuned"
EXP2_METRIC_LOWER_IS_BETTER = True

if not exp2s_i.empty:
    pivot = (exp2s_i.groupby(["variant_name", "model_name"])[EXP2_METRIC_COL]
             .mean().unstack("model_name"))
    pivot["Avg"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("Avg", ascending=EXP2_METRIC_LOWER_IS_BETTER)
    model_cols = [c for c in pivot.columns if c != "Avg"]
    header = " & ".join(["Variant"] + [c.split("/")[-1] for c in model_cols] + ["Avg"]) + r" \\"
    rows_latex = []
    for var, row in pivot.iterrows():
        vals = [row[c] for c in model_cols] + [row["Avg"]]
        formatted = []
        for col, v in zip(model_cols + ["Avg"], vals):
            col_vals = pivot[col].dropna()
            is_winner = (v == col_vals.min()) if EXP2_METRIC_LOWER_IS_BETTER else (v == col_vals.max())
            formatted.append(f"\\textbf{{{v:.3f}}}" if is_winner else f"{v:.3f}")
        rows_latex.append(f"    {var} & " + " & ".join(formatted) + r" \\")
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Jailbreak Tweaks — " + EXP2_METRIC_COL + "}")
    print(r"\begin{tabular}{l" + "r" * (len(model_cols) + 1) + "}")
    print(r"\toprule")
    print(f"    {header}")
    print(r"\midrule")
    for r in rows_latex:
        print(r)
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")
else:
    print("No exp2 data available.")

# NanoGCG vs TROPT GCG (Gemma3) -- step <= MAX_STEPS

All cells below are fed by a single wandb fetch (next cell), capped at
`step <= MAX_STEPS`. They produce, in order:

1. **Full grid** -- per-msg loss curves, time and step axes (5x3 panels).
2. **Paper row** -- 4 randomly sampled msgs, single row, paper-ready.
3. **Bar plot** -- per-msg final loss, mean +/- std across seeds.
4. **Summary** -- avg final loss, win-rate, avg runtime.

In [ ]:
# Single wandb fetch shared by every cell below.
# Builds:
#   dyn_data[x_axis][msg_id][opt] = [(xs, ys), ...]   # for grids/paper-row
#   finals = list[dict(opt, msg_id, seed, final_loss, total_time)]  # for bar/summary
# All values capped at `step <= MAX_STEPS`.
DYN_MODEL_NAME = "google/gemma-3-12b-it"
DYN_OPTIMIZERS = ["gcg", "nanogcg"]
DYN_COLORS     = {"gcg": "#1F4E79", "nanogcg": "#C0392B"}
DYN_LABELS     = {"gcg": "TROPT's GCG", "nanogcg": "nanoGCG's GCG"}
MAX_STEPS      = 500


def _dyn_norm(hist):
    if "step" in hist.columns and "_step" in hist.columns:
        hist = hist.drop(columns=["step"])
    rename = {"_step": "step", "_runtime": "time"}
    return hist.rename(columns={k: v for k, v in rename.items() if k in hist.columns})


def _dyn_curves(hist):
    """{x_axis -> (x, loss)}; truncates to step <= MAX_STEPS."""
    hist = _dyn_norm(hist)
    if "loss" not in hist.columns or "step" not in hist.columns:
        return {}
    hist = hist[hist["step"] <= MAX_STEPS]
    if hist.empty:
        return {}
    out = {}
    for x_col in ("time", "step"):
        if x_col not in hist.columns:
            continue
        v = hist[[x_col, "loss"]].dropna()
        if len(v) < 2:
            continue
        v = v.sort_values(x_col).groupby(x_col, as_index=False)["loss"].mean()
        out[x_col] = (v[x_col].to_numpy(), v["loss"].to_numpy())
    return out


_dyn_api = wandb.Api()
dyn_data: dict[str, dict[int, dict[str, list]]] = {"time": {}, "step": {}}
finals: list[dict] = []
for opt in DYN_OPTIMIZERS:
    runs = list(_dyn_api.runs(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}",
        filters={
            "config.optimizer_name": opt,
            "config.model_name":     DYN_MODEL_NAME,
        },
    ))
    print(f"[{opt}] {len(runs)} run(s)")
    for run in runs:
        cfg = dict(run.config) if run.config else {}
        if not cfg:
            run.load(force=True)
            cfg = dict(run.config)
        msg_id, seed = cfg.get("msg_id"), cfg.get("seed")
        if msg_id is None:
            continue
        hist = run.history(pandas=True)
        if hist.empty:
            continue
        curves = _dyn_curves(hist)
        if not curves:
            continue
        for x_col, (xs, ys) in curves.items():
            (dyn_data[x_col]
             .setdefault(int(msg_id), {})
             .setdefault(opt, [])
             .append((xs, ys)))
        step_curve = curves.get("step")
        time_curve = curves.get("time")
        if step_curve is not None and time_curve is not None:
            finals.append({
                "opt":        opt,
                "msg_id":     int(msg_id),
                "seed":       seed,
                "final_loss": float(step_curve[1][-1]),
                "total_time": float(time_curve[0][-1]),
            })

print(f"\nMsgs covered (time): {sorted(dyn_data['time'].keys())}")
print(f"Msgs covered (step): {sorted(dyn_data['step'].keys())}")
print(f"Final-step records:  {len(finals)}")

In [ ]:
# 5x3 grid: per-msg mean +/- std `loss` curves for time and step axes.
def _dyn_plot_grid(x_axis, x_label, out_pdf):
    msg_ids = sorted(dyn_data[x_axis].keys())
    if not msg_ids:
        print(f"[{x_axis}] no data"); return
    ncols, nrows = 5, 3
    with plt.rc_context({
        "font.family":      "serif",
        "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
    }):
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(3.2 * ncols, 2.4 * nrows),
                                 constrained_layout=True)
        axes = np.array(axes).flatten()
        for ax_idx in range(nrows * ncols):
            ax = axes[ax_idx]
            if ax_idx >= len(msg_ids):
                ax.axis("off"); continue
            msg_id = msg_ids[ax_idx]
            for opt in DYN_OPTIMIZERS:
                runs_data = dyn_data[x_axis][msg_id].get(opt, [])
                if not runs_data:
                    continue
                all_x = np.array(sorted({x for xs, _ in runs_data for x in xs}))
                if len(all_x) < 2:
                    continue
                curves = np.array([np.interp(all_x, xs, ys) for xs, ys in runs_data])
                mean, std = curves.mean(axis=0), curves.std(axis=0)
                c = DYN_COLORS[opt]
                ax.plot(all_x, mean, color=c, linewidth=1.7, label=DYN_LABELS[opt])
                ax.fill_between(all_x, mean - std, mean + std, alpha=0.18, color=c)
            ax.set_yscale("log")
            ax.set_title(f"msg {msg_id}", fontsize=11)
            ax.grid(True, which="both", linestyle=":", linewidth=0.4, color="0.85")
            ax.tick_params(axis="both", labelsize=9)
            if ax_idx % ncols == 0:
                ax.set_ylabel("Loss (per-step)", fontsize=10)
            if ax_idx // ncols == nrows - 1:
                ax.set_xlabel(x_label, fontsize=10)
        # Single shared legend at top.
        for ax in axes[:len(msg_ids)]:
            h, l = ax.get_legend_handles_labels()
            if h:
                fig.legend(h, l, loc="upper center", ncol=2, frameon=False,
                           bbox_to_anchor=(0.5, 1.04), fontsize=12)
                break
        fig.suptitle(
            f"NanoGCG vs TROPT GCG  --  {DYN_MODEL_NAME.split('/')[-1]}  ({x_axis}, <={MAX_STEPS} steps)",
            y=1.08, fontsize=14)
        fig.savefig(out_pdf, bbox_inches="tight", dpi=FIG_DPI)
        plt.show()
        print(f"saved: {out_pdf}")

_dyn_plot_grid("time", "Time (s)", f"{RESULTS_DIR}/fig_dynamics_nanogcg_vs_gcg_grid_time.pdf")
_dyn_plot_grid("step", "Step",     f"{RESULTS_DIR}/fig_dynamics_nanogcg_vs_gcg_grid_step.pdf")

In [ ]:
# Paper-ready single-row: 4 randomly sampled msgs, no titles, linear y,
# legend auto-placed in the freest sub-panel.
PAPER_SAMPLE_SIZE = 4
PAPER_SAMPLE_SEED = 0  # deterministic


def _dyn_plot_paper_row(x_axis, x_label, out_pdf):
    msg_ids_all = sorted(dyn_data[x_axis].keys())
    if len(msg_ids_all) < PAPER_SAMPLE_SIZE:
        sample = msg_ids_all
    else:
        rng = np.random.default_rng(PAPER_SAMPLE_SEED)
        sample = sorted(rng.choice(msg_ids_all, size=PAPER_SAMPLE_SIZE, replace=False).tolist())
    print(f"[paper-{x_axis}] sampled msgs: {sample}")

    with plt.rc_context({
        "font.family":      "serif",
        "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "axes.labelsize":   16,
        "xtick.labelsize":  12,
        "ytick.labelsize":  12,
        "legend.fontsize":  13,
        "axes.linewidth":   0.9,
    }):
        fig, axes = plt.subplots(1, len(sample),
                                 figsize=(3.2 * len(sample), 2.6),
                                 constrained_layout=True, sharey=True)
        if len(sample) == 1:
            axes = np.array([axes])
        for i, msg_id in enumerate(sample):
            ax = axes[i]
            for opt in DYN_OPTIMIZERS:
                runs_data = dyn_data[x_axis][msg_id].get(opt, [])
                if not runs_data:
                    continue
                all_x = np.array(sorted({x for xs, _ in runs_data for x in xs}))
                if len(all_x) < 2:
                    continue
                curves = np.array([np.interp(all_x, xs, ys) for xs, ys in runs_data])
                mean, std = curves.mean(axis=0), curves.std(axis=0)
                c = DYN_COLORS[opt]
                ax.plot(all_x, mean, color=c, linewidth=2.0, label=DYN_LABELS[opt])
                ax.fill_between(all_x, mean - std, mean + std, alpha=0.2, color=c)
            ax.grid(True, which="both", linestyle=":", linewidth=0.4, color="0.85", zorder=0)
            ax.set_axisbelow(True)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
            ax.set_xlabel(x_label)
            if i == 0:
                ax.set_ylabel("Loss")

        # Pick the panel with the freest corner for the legend.
        handles, labels = [], []
        for ax in axes:
            h, l = ax.get_legend_handles_labels()
            if h:
                handles, labels = h, l; break
        if handles:
            best_ax, best_score = axes[0], float("inf")
            for ax in axes:
                leg = ax.legend(handles, labels, loc="best",
                                frameon=True, framealpha=0.92, edgecolor="0.8")
                fig.canvas.draw()
                bbox = leg.get_window_extent()
                overlap = 0
                for line in ax.get_lines():
                    xy = line.get_xydata()
                    if len(xy) == 0:
                        continue
                    disp = ax.transData.transform(xy)
                    overlap += int(((disp[:, 0] >= bbox.x0) & (disp[:, 0] <= bbox.x1) &
                                    (disp[:, 1] >= bbox.y0) & (disp[:, 1] <= bbox.y1)).sum())
                leg.remove()
                if overlap < best_score:
                    best_score, best_ax = overlap, ax
            best_ax.legend(handles, labels, loc="best",
                           frameon=True, framealpha=0.92, edgecolor="0.8")

        fig.savefig(out_pdf, bbox_inches="tight", dpi=FIG_DPI)
        plt.show()
        print(f"saved: {out_pdf}")


_dyn_plot_paper_row("time", "Time (s)", f"{RESULTS_DIR}/fig_dynamics_nanogcg_vs_gcg_paper_time.pdf")
_dyn_plot_paper_row("step", "Step",     f"{RESULTS_DIR}/fig_dynamics_nanogcg_vs_gcg_paper_step.pdf")

In [ ]:
# (iii) Per-message bar plot, mean +/- std across seeds, from `finals`.
if not finals:
    raise RuntimeError("`finals` is empty -- run the fetch cell at the top of this section.")

fin_df = pd.DataFrame(finals)
stats = (fin_df.groupby(["msg_id", "opt"])["final_loss"]
         .agg(["mean", "std", "count"]))
mean_w = stats["mean"].unstack("opt").sort_index()
std_w  = stats["std"].unstack("opt").reindex_like(mean_w).fillna(0.0)

msg_ids = mean_w.index.tolist()
xs      = np.arange(len(msg_ids))
bar_w   = 0.4
ERR_KW  = dict(ecolor="#222222", elinewidth=1.0, capsize=2.5, capthick=1.0)

with plt.rc_context({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize":   18,
    "axes.labelsize":   17,
    "xtick.labelsize":  13,
    "ytick.labelsize":  13,
    "legend.fontsize":  13,
    "axes.linewidth":   0.9,
}):
    fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(msg_ids)), 3.6))
    ax.bar(xs - bar_w / 2, mean_w["nanogcg"].values, width=bar_w,
           yerr=std_w["nanogcg"].values, error_kw=ERR_KW,
           label=DYN_LABELS["nanogcg"], color=DYN_COLORS["nanogcg"],
           edgecolor="white", linewidth=0.7, zorder=2)
    ax.bar(xs + bar_w / 2, mean_w["gcg"].values,     width=bar_w,
           yerr=std_w["gcg"].values,     error_kw=ERR_KW,
           label=DYN_LABELS["gcg"], color=DYN_COLORS["gcg"],
           edgecolor="white", linewidth=0.7, zorder=2)

    ax.set_xticks(xs)
    ax.set_xticklabels(msg_ids)
    ax.set_xlabel("Task ID")
    ax.set_ylabel(r"$\leftarrow$ Final Loss")
    ax.set_xlim(xs[0] - 0.6, xs[-1] + 0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(0.9)
    ax.tick_params(axis="both", direction="out", length=3, width=0.8)
    ax.grid(axis="y", linestyle=":", linewidth=0.6, color="0.7", alpha=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.legend(loc="upper center", frameon=True, framealpha=0.92,
              edgecolor="0.8", fancybox=False, borderpad=0.5,
              handlelength=1.6, handletextpad=0.6)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_nanogcg_vs_gcg_gemma3.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# (iv) Summary: average final loss, win-rate, and average runtime, from `finals`.
if not finals:
    raise RuntimeError("`finals` is empty -- run the fetch cell at the top of this section.")

fin_df = pd.DataFrame(finals)
print(f"{'=' * 70}")
print(f"Summary @ step <= {MAX_STEPS}  --  {DYN_MODEL_NAME}")
print(f"{'=' * 70}")

print("\n(i) Average final loss:")
for opt in DYN_OPTIMIZERS:
    sub = fin_df[fin_df["opt"] == opt]
    if sub.empty:
        continue
    print(f"  {DYN_LABELS[opt]:<18} mean={sub['final_loss'].mean():.4f}  "
          f"std={sub['final_loss'].std():.4f}  n={len(sub)}")

pivot = fin_df.pivot_table(index=["msg_id", "seed"], columns="opt",
                           values="final_loss", aggfunc="first")
paired = pivot.dropna(subset=DYN_OPTIMIZERS)
n = len(paired)
if n == 0:
    print("\n(ii) No paired (msg, seed) tasks for win-rate.")
else:
    a, b = DYN_OPTIMIZERS
    wins = {a: int((paired[a] < paired[b]).sum()),
            b: int((paired[b] < paired[a]).sum())}
    ties = int((paired[a] == paired[b]).sum())
    print(f"\n(ii) Win-rate over {n} paired (msg, seed) tasks:")
    for opt in DYN_OPTIMIZERS:
        print(f"  {DYN_LABELS[opt]:<18} wins={wins[opt]:>3}  ({wins[opt] / n:.1%})")
    print(f"  {'Ties':<18} ties={ties:>3}  ({ties / n:.1%})")

print("\n(iii) Average time to MAX_STEPS:")
for opt in DYN_OPTIMIZERS:
    sub = fin_df[fin_df["opt"] == opt]
    if sub.empty:
        continue
    print(f"  {DYN_LABELS[opt]:<18} mean={sub['total_time'].mean():.1f}s  "
          f"std={sub['total_time'].std():.1f}s  n={len(sub)}")
print(f"{'=' * 70}")